# Punjab Prior-Constrained Inversion

This notebook is the working surface for the Punjab inversion stage.

It builds on the reusable modules extracted from the synthetic benchmark notebook and is intended for the prior-constrained formulation used in the final real-data workflow.

## Workflow

The notebook is organized as:

1. Environment and shared imports
2. Punjab data loading and alignment
3. Generic-prior definition for the first Punjab inversion
4. Model setup
5. Training with forward, spatial, and temporal constraints
6. Evaluation and export
7. External-prior extension hooks for a later GRACE/W3RA/GPS phase

In [ ]:
from pathlib import Path
import json
import math
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
import xarray as xr

from punjab_inversion import (
    PhysicsConfig,
    build_fft_kernels,
    forward_physics_torch,
    forward_two_layer_torch,
    DualDecoderFrequencySeparatedSwinUNet3D,
    NoiseConditionedDualDecoderSwinUNet3D,
    normalize_field,
    batch_correlation_torch,
    amplitude_penalty_torch,
    anisotropic_total_variation,
    rmse,
    r2_score_np,
    corr_np,
    mae,
    bias_np,
    nrmse_np,
    fit_slope_np,
    fit_intercept_np,
    set_seed,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ROOT = Path.cwd()
OUT_DIR = ROOT / 'outputs' / 'punjab_prior'
FIG_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Device:', DEVICE)
print('Root:', ROOT)


In [ ]:
# Core run configuration
SEED = 42
set_seed(SEED)

PHYSICS = PhysicsConfig()

WINDOW_SIZE = 12
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 30
FORWARD_NORMALIZE = True

# Revised phase-1 support strategy: use temporal support plus a relaxed coherence threshold.
SUPPORT_CONFIG = {
    'mask_strategy': 'coherence_and_temporal_support',
    'coherence_threshold': 0.20,
    'time_valid_fraction_threshold': 0.05,
    'tile_size': (64, 64),
    'tile_stride': (64, 64),
    'tile_min_valid_fraction': 0.20,
}

# Bounded pilot strategy: keep revised support coverage, but group samples by tile so
# the temporal prior sees consecutive windows from the same tile within the pilot budget.
PILOT_GROUP_CONFIG = {
    'strategy': 'tile_grouped_temporal_progression',
    'train_tile_count': 4,
    'val_tile_count': 4,
    'train_windows_per_tile': 16,
    'val_windows_per_tile': 8,
}

EXPANDED_PILOT_GROUP_CONFIG = {
    'strategy': 'tile_grouped_temporal_progression_expanded',
    'train_tile_count': 6,
    'val_tile_count': 6,
    'train_windows_per_tile': 20,
    'val_windows_per_tile': 10,
}

FULLER_PHASE1_CONFIG = {
    'strategy': 'phase1_fuller_grouped_run',
    'train_tile_count': 8,
    'val_tile_count': 8,
    'train_windows_per_tile': 24,
    'val_windows_per_tile': 12,
    'num_epochs': 6,
}

FINETUNE_PHASE1_CONFIG = {
    'strategy': 'phase1_fuller_warmstart_finetune',
    'train_tile_count': 8,
    'val_tile_count': 8,
    'train_windows_per_tile': 24,
    'val_windows_per_tile': 12,
    'num_epochs': 6,
    'learning_rate': 5e-5,
    'init_from': 'grouped_support_expanded',
}

# Phase 1 uses only generic priors. External priors are deferred.
PRIOR_WEIGHTS = {
    'forward': 1.0,
    'spatial_s0': 1e-3,
    'spatial_sg': 1e-3,
    'temporal_s0': 5e-4,
    'temporal_sg': 5e-4,
    'hydrology': 0.0,
    'grace': 0.0,
    'gps': 0.0,
}

PRIOR_CONFIG = {
    'phase': 'generic_priors_only',
    'use_noise_conditioning': False,
    'use_spatial_prior': True,
    'use_temporal_prior': True,
    'use_grace_prior': False,
    'use_gps_prior': False,
    'use_hydrology_prior': False,
}

print(json.dumps(PRIOR_WEIGHTS, indent=2))
print(json.dumps(PILOT_GROUP_CONFIG, indent=2))
print(json.dumps(EXPANDED_PILOT_GROUP_CONFIG, indent=2))
print(json.dumps(FULLER_PHASE1_CONFIG, indent=2))
print(json.dumps(FINETUNE_PHASE1_CONFIG, indent=2))





## Punjab Data Loading And Alignment

Load the Punjab deformation data and the core grid/time metadata here.

For this first inversion pass we do not depend on GRACE, W3RA, or GPS. Those hooks stay deferred until the generic-prior baseline is in place.

Expected outputs of this section:

- aligned deformation tensor
- aligned masks and static grids
- optional noise proxy channel if we decide to re-enable conditioning later
- train/validation/test temporal splits

In [ ]:
# TODO: replace these placeholders with the real Punjab data sources.
DATA_PATHS = {
    'deformation_root': '/mnt/data/aoi_punjab',
    'mask': None,
    'static': None,
    'w3ra': str(ROOT / 'outputs' / 'W3RA_punjab_preclipped.nc'),
}

def parse_acquisition_dates(date_path):
    with h5py.File(date_path, 'r') as f:
        raw = f['acquisition_dates'][0]
    return pd.to_datetime([v.decode() if isinstance(v, bytes) else str(v) for v in raw], format='%d-%b-%Y')


def load_punjab_stack(data_paths, load_full=False):
    root = Path(data_paths['deformation_root'])
    disp_path = root / 'disp_all_ll.h5'
    coh_path = root / 'coh_ll.h5'
    vel_path = root / 'vel_ll.h5'
    date_path = root / 'aquisition_dates_ll.h5'

    dates = parse_acquisition_dates(date_path)
    with h5py.File(disp_path, 'r') as f_disp, h5py.File(coh_path, 'r') as f_coh, h5py.File(vel_path, 'r') as f_vel:
        lat = np.asarray(f_disp['lat'])
        lon = np.asarray(f_disp['lon'])
        disp_shape = tuple(f_disp['z'].shape)
        coh_shape = tuple(f_coh['z'].shape)
        vel_shape = tuple(f_vel['z'].shape)
        stack = {
            'root': str(root),
            'disp_path': str(disp_path),
            'coh_path': str(coh_path),
            'vel_path': str(vel_path),
            'date_path': str(date_path),
            'dates': dates,
            'lat': lat,
            'lon': lon,
            'disp_shape': disp_shape,
            'coh_shape': coh_shape,
            'vel_shape': vel_shape,
        }
        if load_full:
            stack['disp'] = np.asarray(f_disp['z'])
            stack['coh'] = np.asarray(f_coh['z'])
            stack['vel'] = np.asarray(f_vel['z'])
    return stack


def load_w3ra_reference(data_paths):
    w3ra_path = Path(data_paths['w3ra'])
    return xr.open_dataset(w3ra_path)


punjab_meta = load_punjab_stack(DATA_PATHS, load_full=False)
w3ra_ref = load_w3ra_reference(DATA_PATHS)

punjab_summary = pd.DataFrame([
    {
        'deformation_root': punjab_meta['root'],
        'n_times': punjab_meta['disp_shape'][0],
        'n_lat': punjab_meta['disp_shape'][1],
        'n_lon': punjab_meta['disp_shape'][2],
        'date_start': punjab_meta['dates'].min(),
        'date_end': punjab_meta['dates'].max(),
        'w3ra_start': pd.to_datetime(w3ra_ref.time.values[0]),
        'w3ra_end': pd.to_datetime(w3ra_ref.time.values[-1]),
        'w3ra_shape': tuple(w3ra_ref['S0'].shape),
    }
])
print(punjab_summary.to_string(index=False))
print('Punjab lat/lon lengths:', len(punjab_meta['lat']), len(punjab_meta['lon']))
print('W3RA variables:', list(w3ra_ref.data_vars))

## Coverage Diagnostics

Before building a training dataset, inspect how much of the Punjab grid is actually usable. The deformation stack is sparse at the full-grid level, so training should be patch-based and mask-aware.

In [ ]:
def compute_time_valid_fraction(disp_path, chunk_size=25):
    with h5py.File(disp_path, 'r') as f_disp:
        z = f_disp['z']
        counts = np.zeros(z.shape[1:], dtype=np.uint16)
        n_time = z.shape[0]
        for t0 in range(0, n_time, chunk_size):
            chunk = np.asarray(z[t0:min(t0 + chunk_size, n_time)])
            counts += np.isfinite(chunk).sum(axis=0).astype(np.uint16)
    return counts.astype(np.float32) / max(n_time, 1)


def summarize_punjab_coverage(meta, support_cfg, spatial_stride=40):
    with h5py.File(meta['disp_path'], 'r') as f_disp, h5py.File(meta['coh_path'], 'r') as f_coh, h5py.File(meta['vel_path'], 'r') as f_vel:
        disp_sample = np.asarray(f_disp['z'][:, ::spatial_stride, ::spatial_stride])
        coh_full = np.asarray(f_coh['z'])
        vel_full = np.asarray(f_vel['z'])
    time_valid_fraction = compute_time_valid_fraction(meta['disp_path'])

    disp_sample_finite = np.isfinite(disp_sample)
    per_time_valid = disp_sample_finite.reshape(disp_sample_finite.shape[0], -1).mean(axis=1)
    coh_valid = np.isfinite(coh_full)
    vel_valid = np.isfinite(vel_full)
    temporal_support = time_valid_fraction >= support_cfg['time_valid_fraction_threshold']
    support_mask = coh_valid & temporal_support & (coh_full >= support_cfg['coherence_threshold'])

    def count_tiles(mask, tile_size, min_valid):
        count = 0
        for row in range(0, mask.shape[0] - tile_size + 1, tile_size):
            for col in range(0, mask.shape[1] - tile_size + 1, tile_size):
                if mask[row:row + tile_size, col:col + tile_size].mean() >= min_valid:
                    count += 1
        return count

    strategy_rows = []
    candidate_masks = {
        'current_velocity_and_coherence': coh_valid & vel_valid & (coh_full >= 0.30),
        'temporal_support_only': temporal_support,
        'coherence_and_temporal_support': support_mask,
        'relaxed_coherence_temporal': coh_valid & (coh_full >= 0.15) & temporal_support,
    }
    for name, mask in candidate_masks.items():
        for tile_size in [64, 96, 128]:
            for min_valid in [0.10, 0.20, 0.25, 0.30]:
                strategy_rows.append({
                    'strategy': name,
                    'mask_fraction': float(mask.mean()),
                    'tile_size': tile_size,
                    'tile_min_valid_fraction': min_valid,
                    'tile_count': count_tiles(mask, tile_size, min_valid),
                })
    strategy_df = pd.DataFrame(strategy_rows)

    summary = {
        'coherence_threshold': support_cfg['coherence_threshold'],
        'time_valid_fraction_threshold': support_cfg['time_valid_fraction_threshold'],
        'sample_stride': spatial_stride,
        'disp_sample_valid_fraction': float(disp_sample_finite.mean()),
        'disp_per_time_valid_min': float(per_time_valid.min()),
        'disp_per_time_valid_max': float(per_time_valid.max()),
        'disp_per_time_valid_mean': float(per_time_valid.mean()),
        'time_valid_fraction_mean': float(time_valid_fraction.mean()),
        'coh_valid_fraction': float(coh_valid.mean()),
        'vel_valid_fraction': float(vel_valid.mean()),
        'support_mask_fraction': float(support_mask.mean()),
        'coh_min': float(np.nanmin(coh_full)),
        'coh_max': float(np.nanmax(coh_full)),
        'vel_min': float(np.nanmin(vel_full)),
        'vel_max': float(np.nanmax(vel_full)),
    }
    return summary, support_mask, coh_full, vel_full, time_valid_fraction, strategy_df


coverage_summary, support_mask, coh_full, vel_full, time_valid_fraction, strategy_df = summarize_punjab_coverage(punjab_meta, SUPPORT_CONFIG)
coverage_df = pd.DataFrame([coverage_summary])
print(coverage_df.to_string(index=False))
print(strategy_df.sort_values(['strategy', 'tile_size', 'tile_min_valid_fraction']).head(24).to_string(index=False))

coverage_df.to_csv(OUT_DIR / 'punjab_phase1_coverage_summary.csv', index=False)
strategy_df.to_csv(OUT_DIR / 'punjab_phase1_support_strategy_sweep.csv', index=False)
with open(OUT_DIR / 'punjab_phase1_coverage_summary.json', 'w') as f:
    json.dump(coverage_summary, f, indent=2)

fig, axes = plt.subplots(1, 4, figsize=(19, 4))
axes[0].imshow(support_mask, cmap='gray')
axes[0].set_title('Selected Support Mask')
axes[1].imshow(coh_full, cmap='viridis')
axes[1].set_title('Coherence')
axes[2].imshow(time_valid_fraction, cmap='magma', vmin=0, vmax=1)
axes[2].set_title('Time Valid Fraction')
axes[3].imshow(vel_full, cmap='coolwarm')
axes[3].set_title('Velocity')
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.savefig(FIG_DIR / 'punjab_phase1_coverage_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## Chronological Splits And Patch-Based Dataset

The full Punjab grid is too large for direct end-to-end training with the current model. The first runnable training unit should therefore be a temporal window over a spatial tile cut from the valid support mask.

In [ ]:
def make_chronological_splits(dates, train_frac=0.70, val_frac=0.15, window_size=12):
    n_total = len(dates)
    valid_end_idx = np.arange(window_size - 1, n_total)
    n_windows = len(valid_end_idx)
    n_train = int(n_windows * train_frac)
    n_val = int(n_windows * val_frac)
    train_idx = valid_end_idx[:n_train]
    val_idx = valid_end_idx[n_train:n_train + n_val]
    test_idx = valid_end_idx[n_train + n_val:]
    return {
        'train': train_idx,
        'val': val_idx,
        'test': test_idx,
    }


def build_valid_tiles(valid_mask, tile_size=(64, 64), stride=(64, 64), min_valid_fraction=0.25):
    tiles = []
    tile_h, tile_w = tile_size
    stride_h, stride_w = stride
    h, w = valid_mask.shape
    for row in range(0, h - tile_h + 1, stride_h):
        for col in range(0, w - tile_w + 1, stride_w):
            tile = valid_mask[row:row + tile_h, col:col + tile_w]
            frac = float(tile.mean())
            if frac >= min_valid_fraction:
                tiles.append({'row': row, 'col': col, 'valid_fraction': frac})
    return tiles


class PunjabWindowedTileDataset(Dataset):
    def __init__(self, disp_path, end_indices, tiles, support_mask, window_size=12, fill_value=0.0):
        self.disp_path = str(disp_path)
        self.end_indices = list(end_indices)
        self.tiles = list(tiles)
        self.support_mask = support_mask.astype(np.float32)
        self.window_size = int(window_size)
        self.fill_value = float(fill_value)

    def __len__(self):
        return len(self.end_indices) * len(self.tiles)

    def __getitem__(self, idx):
        time_idx = idx // len(self.tiles)
        tile_idx = idx % len(self.tiles)
        end_idx = self.end_indices[time_idx]
        start_idx = end_idx - self.window_size + 1
        tile = self.tiles[tile_idx]
        r0, c0 = tile['row'], tile['col']
        r1, c1 = r0 + 64, c0 + 64

        with h5py.File(self.disp_path, 'r') as f_disp:
            x = np.asarray(f_disp['z'][start_idx:end_idx + 1, r0:r1, c0:c1], dtype=np.float32)
        mask = self.support_mask[r0:r1, c0:c1]
        x = np.where(np.isfinite(x), x, self.fill_value)
        x = x * mask[None, ...]
        last_disp = x[-1][None, ...]
        return {
            'x': torch.tensor(x[None, ...], dtype=torch.float32),
            'obs_disp': torch.tensor(last_disp, dtype=torch.float32),
            'mask': torch.tensor(mask[None, ...], dtype=torch.float32),
            'end_idx': int(end_idx),
            'tile_row': int(r0),
            'tile_col': int(c0),
        }


split_idx = make_chronological_splits(punjab_meta['dates'], window_size=WINDOW_SIZE)
tiles = build_valid_tiles(
    support_mask,
    tile_size=SUPPORT_CONFIG['tile_size'],
    stride=SUPPORT_CONFIG['tile_stride'],
    min_valid_fraction=SUPPORT_CONFIG['tile_min_valid_fraction'],
)

split_summary = pd.DataFrame([
    {'split': 'train', 'n_windows': len(split_idx['train']), 'date_start': punjab_meta['dates'][split_idx['train'][0]], 'date_end': punjab_meta['dates'][split_idx['train'][-1]]},
    {'split': 'val', 'n_windows': len(split_idx['val']), 'date_start': punjab_meta['dates'][split_idx['val'][0]], 'date_end': punjab_meta['dates'][split_idx['val'][-1]]},
    {'split': 'test', 'n_windows': len(split_idx['test']), 'date_start': punjab_meta['dates'][split_idx['test'][0]], 'date_end': punjab_meta['dates'][split_idx['test'][-1]]},
])
print(split_summary.to_string(index=False))
print('support config:', SUPPORT_CONFIG)
print('usable tiles:', len(tiles))
if tiles:
    print('first tile:', tiles[0])

split_summary.to_csv(OUT_DIR / 'punjab_phase1_split_summary.csv', index=False)
pd.DataFrame(tiles).to_csv(OUT_DIR / 'punjab_phase1_tile_inventory.csv', index=False)

train_ds = PunjabWindowedTileDataset(punjab_meta['disp_path'], split_idx['train'], tiles, support_mask, window_size=WINDOW_SIZE)
sample = train_ds[0] if len(train_ds) else None
if sample is not None:
    print('sample x shape:', tuple(sample['x'].shape))
    print('sample obs_disp shape:', tuple(sample['obs_disp'].shape))
    print('sample mask coverage:', float(sample['mask'].mean()))

## Normalization And DataLoaders

Normalize the deformation input using training tiles only. The observed deformation target for the forward loss stays in physical units.

In [ ]:
NORM_BATCH_SIZE = 4
NUM_WORKERS = 0


def compute_scalar_stats(dataset, key, batch_size=NORM_BATCH_SIZE, max_batches=None):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    total_sum = 0.0
    total_sq = 0.0
    total_count = 0
    for batch_idx, batch in enumerate(loader):
        x = batch[key].float()
        total_sum += x.sum().item()
        total_sq += (x * x).sum().item()
        total_count += x.numel()
        if max_batches is not None and (batch_idx + 1) >= max_batches:
            break
    mean = total_sum / max(total_count, 1)
    var = max(total_sq / max(total_count, 1) - mean ** 2, 1e-8)
    std = float(np.sqrt(var))
    return mean, std


class NormalizedPunjabTileDataset(Dataset):
    def __init__(self, base_dataset, x_mean, x_std):
        self.base_dataset = base_dataset
        self.x_mean = float(x_mean)
        self.x_std = float(max(x_std, 1e-6))

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        return {
            **item,
            'x': (item['x'] - self.x_mean) / self.x_std,
        }


def select_evenly_spaced_tile_indices(n_tiles, tile_count):
    if n_tiles <= 0 or tile_count <= 0:
        return []
    tile_count = min(int(tile_count), int(n_tiles))
    if tile_count == 1:
        return [0]
    indices = np.linspace(0, n_tiles - 1, tile_count)
    return sorted({int(round(v)) for v in indices})


def build_tile_grouped_linear_indices(end_indices, tiles, tile_count, windows_per_tile):
    n_times = len(end_indices)
    n_tiles = len(tiles)
    chosen_tiles = select_evenly_spaced_tile_indices(n_tiles, tile_count)
    chosen_times = list(range(min(int(windows_per_tile), n_times)))
    linear_indices = []
    for tile_idx in chosen_tiles:
        for time_idx in chosen_times:
            linear_indices.append(time_idx * n_tiles + tile_idx)
    return linear_indices, chosen_tiles, chosen_times


train_raw_ds = PunjabWindowedTileDataset(punjab_meta['disp_path'], split_idx['train'], tiles, support_mask, window_size=WINDOW_SIZE)
val_raw_ds = PunjabWindowedTileDataset(punjab_meta['disp_path'], split_idx['val'], tiles, support_mask, window_size=WINDOW_SIZE)
test_raw_ds = PunjabWindowedTileDataset(punjab_meta['disp_path'], split_idx['test'], tiles, support_mask, window_size=WINDOW_SIZE)

x_mean, x_std = compute_scalar_stats(train_raw_ds, key='x')
obs_mean, obs_std = compute_scalar_stats(train_raw_ds, key='obs_disp')
disp_mean_t = torch.tensor(obs_mean, dtype=torch.float32, device=DEVICE).view(1, 1, 1, 1)
disp_std_t = torch.tensor(max(obs_std, 1e-6), dtype=torch.float32, device=DEVICE).view(1, 1, 1, 1)

train_ds = NormalizedPunjabTileDataset(train_raw_ds, x_mean, x_std)
val_ds = NormalizedPunjabTileDataset(val_raw_ds, x_mean, x_std)
test_ds = NormalizedPunjabTileDataset(test_raw_ds, x_mean, x_std)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

pilot_train_indices, pilot_train_tiles, pilot_train_times = build_tile_grouped_linear_indices(
    split_idx['train'],
    tiles,
    tile_count=PILOT_GROUP_CONFIG['train_tile_count'],
    windows_per_tile=PILOT_GROUP_CONFIG['train_windows_per_tile'],
)
pilot_val_indices, pilot_val_tiles, pilot_val_times = build_tile_grouped_linear_indices(
    split_idx['val'],
    tiles,
    tile_count=PILOT_GROUP_CONFIG['val_tile_count'],
    windows_per_tile=PILOT_GROUP_CONFIG['val_windows_per_tile'],
)
expanded_train_indices, expanded_train_tiles, expanded_train_times = build_tile_grouped_linear_indices(
    split_idx['train'],
    tiles,
    tile_count=EXPANDED_PILOT_GROUP_CONFIG['train_tile_count'],
    windows_per_tile=EXPANDED_PILOT_GROUP_CONFIG['train_windows_per_tile'],
)
expanded_val_indices, expanded_val_tiles, expanded_val_times = build_tile_grouped_linear_indices(
    split_idx['val'],
    tiles,
    tile_count=EXPANDED_PILOT_GROUP_CONFIG['val_tile_count'],
    windows_per_tile=EXPANDED_PILOT_GROUP_CONFIG['val_windows_per_tile'],
)

pilot_train_ds = Subset(train_ds, pilot_train_indices)
pilot_val_ds = Subset(val_ds, pilot_val_indices)
pilot_train_loader = DataLoader(pilot_train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
pilot_val_loader = DataLoader(pilot_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

expanded_pilot_train_ds = Subset(train_ds, expanded_train_indices)
expanded_pilot_val_ds = Subset(val_ds, expanded_val_indices)
expanded_pilot_train_loader = DataLoader(expanded_pilot_train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
expanded_pilot_val_loader = DataLoader(expanded_pilot_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

fuller_train_indices, fuller_train_tiles, fuller_train_times = build_tile_grouped_linear_indices(
    split_idx['train'],
    tiles,
    tile_count=FULLER_PHASE1_CONFIG['train_tile_count'],
    windows_per_tile=FULLER_PHASE1_CONFIG['train_windows_per_tile'],
)
fuller_val_indices, fuller_val_tiles, fuller_val_times = build_tile_grouped_linear_indices(
    split_idx['val'],
    tiles,
    tile_count=FULLER_PHASE1_CONFIG['val_tile_count'],
    windows_per_tile=FULLER_PHASE1_CONFIG['val_windows_per_tile'],
)

fuller_train_ds = Subset(train_ds, fuller_train_indices)
fuller_val_ds = Subset(val_ds, fuller_val_indices)
fuller_train_loader = DataLoader(fuller_train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
fuller_val_loader = DataLoader(fuller_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

norm_summary = {
    'x_mean': float(x_mean),
    'x_std': float(x_std),
    'obs_mean': float(obs_mean),
    'obs_std': float(obs_std),
    'n_train_samples': len(train_ds),
    'n_val_samples': len(val_ds),
    'n_test_samples': len(test_ds),
    'batch_size': BATCH_SIZE,
    'pilot_group_strategy': PILOT_GROUP_CONFIG['strategy'],
    'pilot_train_group_samples': len(pilot_train_ds),
    'pilot_val_group_samples': len(pilot_val_ds),
    'pilot_train_tile_indices': pilot_train_tiles,
    'pilot_val_tile_indices': pilot_val_tiles,
    'pilot_train_time_span': [int(t) for t in pilot_train_times],
    'pilot_val_time_span': [int(t) for t in pilot_val_times],
    'expanded_pilot_group_strategy': EXPANDED_PILOT_GROUP_CONFIG['strategy'],
    'expanded_pilot_train_group_samples': len(expanded_pilot_train_ds),
    'expanded_pilot_val_group_samples': len(expanded_pilot_val_ds),
    'expanded_pilot_train_tile_indices': expanded_train_tiles,
    'expanded_pilot_val_tile_indices': expanded_val_tiles,
    'expanded_pilot_train_time_span': [int(t) for t in expanded_train_times],
    'expanded_pilot_val_time_span': [int(t) for t in expanded_val_times],
    'fuller_phase1_strategy': FULLER_PHASE1_CONFIG['strategy'],
    'fuller_train_group_samples': len(fuller_train_ds),
    'fuller_val_group_samples': len(fuller_val_ds),
    'fuller_train_tile_indices': fuller_train_tiles,
    'fuller_val_tile_indices': fuller_val_tiles,
    'fuller_train_time_span': [int(t) for t in fuller_train_times],
    'fuller_val_time_span': [int(t) for t in fuller_val_times],
    'finetune_phase1_strategy': FINETUNE_PHASE1_CONFIG['strategy'],
    'finetune_init_from': FINETUNE_PHASE1_CONFIG['init_from'],
    'finetune_learning_rate': FINETUNE_PHASE1_CONFIG['learning_rate'],
}

print(json.dumps(norm_summary, indent=2))
with open(OUT_DIR / 'punjab_phase1_normalization.json', 'w') as f:
    json.dump(norm_summary, f, indent=2)





## Generic Priors For The First Punjab Inversion

The first Punjab inversion uses only generic regularization terms. External priors such as GRACE, W3RA, and GPS are intentionally deferred to a later integration phase.

The phase-1 objective is:

$$L_{phase1} = \lambda_f L_{fwd} + \lambda_{s0} L_{spatial}^{S_0} + \lambda_{sg} L_{spatial}^{S_g} + \lambda_{t0} L_{temporal}^{S_0} + \lambda_{tg} L_{temporal}^{S_g}$$

where the forward term keeps the inversion tied to deformation physics, the spatial term suppresses map-scale artifacts, and the temporal term discourages unrealistic month-to-month jumps.

In [ ]:
def spatial_prior_loss(pred):
    return anisotropic_total_variation(pred)


def temporal_prior_loss(current_pred, previous_pred=None):
    if previous_pred is None:
        return torch.tensor(0.0, device=current_pred.device)
    return torch.abs(current_pred - previous_pred).mean()


def split_two_layer_prediction(pred_state):
    return pred_state[:, 0:1], pred_state[:, 1:2]


def build_generic_prior_terms(pred_state, previous_state=None):
    s0_pred, sg_pred = split_two_layer_prediction(pred_state)
    prev_s0 = None if previous_state is None else previous_state[:, 0:1]
    prev_sg = None if previous_state is None else previous_state[:, 1:2]
    return {
        'spatial_s0': spatial_prior_loss(s0_pred),
        'spatial_sg': spatial_prior_loss(sg_pred),
        'temporal_s0': temporal_prior_loss(s0_pred, prev_s0),
        'temporal_sg': temporal_prior_loss(sg_pred, prev_sg),
    }


def build_external_prior_terms(pred_state, pred_disp, targets):
    prior_terms = {}
    if PRIOR_CONFIG['use_hydrology_prior'] and 'hydrology' in targets:
        prior_terms['hydrology'] = F.mse_loss(pred_state, targets['hydrology'])
    if PRIOR_CONFIG['use_grace_prior'] and 'grace' in targets:
        prior_terms['grace'] = F.mse_loss(pred_state.sum(dim=1, keepdim=True), targets['grace'])
    if PRIOR_CONFIG['use_gps_prior'] and 'gps' in targets:
        prior_terms['gps'] = F.mse_loss(pred_disp, targets['gps'])
    return prior_terms

## Model Setup

Start from the best synthetic clean-anchor architecture. We can re-introduce conditioning later if we build a real Punjab noise proxy.

In [ ]:
MODEL_KIND = 'dual_decoder_frequency'

if MODEL_KIND == 'noise_conditioned_dual_decoder':
    model = NoiseConditionedDualDecoderSwinUNet3D(
        base_dim=32,
        time_patch=2,
        spatial_patch=4,
        num_heads=4,
    ).to(DEVICE)
else:
    model = DualDecoderFrequencySeparatedSwinUNet3D(
        base_dim=32,
        time_patch=2,
        spatial_patch=4,
        num_heads=4,
    ).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print(model.__class__.__name__)
sum(p.numel() for p in model.parameters())

## Training And Inversion

This section implements the first Punjab run with only generic priors: forward consistency plus spatial and temporal regularization.

In [ ]:
def build_phase1_loss(pred_state, pred_disp, obs_disp, weights, previous_state=None, external_targets=None, disp_mean=None, disp_std=None):
    generic_terms = build_generic_prior_terms(pred_state, previous_state=previous_state)
    external_terms = build_external_prior_terms(pred_state, pred_disp, external_targets or {})
    if disp_mean is not None and disp_std is not None and FORWARD_NORMALIZE:
        pred_disp_cmp = normalize_field(pred_disp, disp_mean, disp_std)
        obs_disp_cmp = normalize_field(obs_disp, disp_mean, disp_std)
    else:
        pred_disp_cmp = pred_disp
        obs_disp_cmp = obs_disp
    loss_forward = F.mse_loss(pred_disp_cmp, obs_disp_cmp)
    total = (
        weights['forward'] * loss_forward
        + weights['spatial_s0'] * generic_terms['spatial_s0']
        + weights['spatial_sg'] * generic_terms['spatial_sg']
        + weights['temporal_s0'] * generic_terms['temporal_s0']
        + weights['temporal_sg'] * generic_terms['temporal_sg']
        + weights['hydrology'] * external_terms.get('hydrology', torch.tensor(0.0, device=pred_state.device))
        + weights['grace'] * external_terms.get('grace', torch.tensor(0.0, device=pred_state.device))
        + weights['gps'] * external_terms.get('gps', torch.tensor(0.0, device=pred_state.device))
    )
    return {
        'total': total,
        'forward': loss_forward,
        **generic_terms,
        **external_terms,
    }


def run_phase1_epoch(model, loader, optimizer=None, *, physics_cfg, green_load_fft, green_poro_fft, weights, device=DEVICE, max_batches=None, disp_mean=None, disp_std=None):
    training = optimizer is not None
    model.train(mode=training)
    state_cache = {}
    stats = {
        'total': 0.0,
        'forward': 0.0,
        'spatial_s0': 0.0,
        'spatial_sg': 0.0,
        'temporal_s0': 0.0,
        'temporal_sg': 0.0,
        'n_samples': 0,
    }

    for batch_idx, batch in enumerate(loader):
        xb = batch['x'].to(device)
        obs_disp = batch['obs_disp'].to(device)
        mask = batch['mask'].to(device)
        tile_rows = batch['tile_row'].tolist()
        tile_cols = batch['tile_col'].tolist()

        with torch.set_grad_enabled(training):
            pred_state = model(xb)
            batch_losses = []
            cache_updates = []

            for i in range(pred_state.shape[0]):
                key = (int(tile_rows[i]), int(tile_cols[i]))
                prev_state = state_cache.get(key)
                pred_i = pred_state[i:i+1] * mask[i:i+1]
                pred_disp_i = forward_two_layer_torch(pred_i, green_load_fft, green_poro_fft, physics_cfg) * mask[i:i+1]
                obs_i = obs_disp[i:i+1]
                losses = build_phase1_loss(pred_i, pred_disp_i, obs_i, weights, previous_state=prev_state, disp_mean=disp_mean, disp_std=disp_std)
                batch_losses.append(losses['total'])
                for name in ['total', 'forward', 'spatial_s0', 'spatial_sg', 'temporal_s0', 'temporal_sg']:
                    stats[name] += float(losses[name].detach().cpu())
                stats['n_samples'] += 1
                cache_updates.append((key, pred_i.detach()))

            loss = torch.stack(batch_losses).mean()
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        for key, cached_pred in cache_updates:
            state_cache[key] = cached_pred

        if max_batches is not None and (batch_idx + 1) >= max_batches:
            break

    denom = max(stats['n_samples'], 1)
    return {k: (v / denom if k != 'n_samples' else v) for k, v in stats.items()}


tile_h = int(sample['obs_disp'].shape[-2])
tile_w = int(sample['obs_disp'].shape[-1])
g_load_fft_t, g_poro_fft_t = build_fft_kernels(tile_h, tile_w, PHYSICS, DEVICE)


def run_phase1_smoke_train(num_epochs=2, train_batches=4, val_batches=2):
    smoke_model = DualDecoderFrequencySeparatedSwinUNet3D(
        base_dim=32,
        time_patch=2,
        spatial_patch=4,
        num_heads=4,
    ).to(DEVICE)
    smoke_optim = torch.optim.AdamW(smoke_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history = []
    for epoch in range(num_epochs):
        train_stats = run_phase1_epoch(
            smoke_model,
            train_loader,
            optimizer=smoke_optim,
            physics_cfg=PHYSICS,
            green_load_fft=g_load_fft_t,
            green_poro_fft=g_poro_fft_t,
            weights=PRIOR_WEIGHTS,
            max_batches=train_batches,
            disp_mean=disp_mean_t,
            disp_std=disp_std_t,
        )
        val_stats = run_phase1_epoch(
            smoke_model,
            val_loader,
            optimizer=None,
            physics_cfg=PHYSICS,
            green_load_fft=g_load_fft_t,
            green_poro_fft=g_poro_fft_t,
            weights=PRIOR_WEIGHTS,
            max_batches=val_batches,
            disp_mean=disp_mean_t,
            disp_std=disp_std_t,
        )
        row = {'epoch': epoch + 1}
        row.update({f'train_{k}': v for k, v in train_stats.items()})
        row.update({f'val_{k}': v for k, v in val_stats.items()})
        history.append(row)
        print(row)

    history_df = pd.DataFrame(history)
    history_path = OUT_DIR / 'punjab_phase1_smoke_history.csv'
    history_df.to_csv(history_path, index=False)
    summary_path = OUT_DIR / 'punjab_phase1_smoke_summary.json'
    with open(summary_path, 'w') as f:
        json.dump(history_df.iloc[-1].to_dict(), f, indent=2)
    return smoke_model, history_df, history_path, summary_path


# Run a small end-to-end smoke test before a full Punjab inversion.
smoke_model, smoke_history_df, smoke_history_path, smoke_summary_path = run_phase1_smoke_train(num_epochs=2, train_batches=4, val_batches=2)
print('Saved smoke history to:', smoke_history_path)
print('Saved smoke summary to:', smoke_summary_path)


def run_phase1_pilot_train(*, train_loader_pilot, val_loader_pilot, label='grouped_support', pilot_group_config=None, num_epochs=4, init_checkpoint=None, learning_rate=LEARNING_RATE):
    pilot_model = DualDecoderFrequencySeparatedSwinUNet3D(
        base_dim=32,
        time_patch=2,
        spatial_patch=4,
        num_heads=4,
    ).to(DEVICE)
    pilot_optim = torch.optim.AdamW(pilot_model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    if init_checkpoint is not None and Path(init_checkpoint).exists():
        ckpt = torch.load(init_checkpoint, map_location=DEVICE)
        pilot_model.load_state_dict(ckpt['model_state_dict'])
    history = []
    best_val = float('inf')
    best_ckpt_path = OUT_DIR / f'punjab_phase1_pilot_best_{label}.pt'
    for epoch in range(num_epochs):
        train_stats = run_phase1_epoch(
            pilot_model,
            train_loader_pilot,
            optimizer=pilot_optim,
            physics_cfg=PHYSICS,
            green_load_fft=g_load_fft_t,
            green_poro_fft=g_poro_fft_t,
            weights=PRIOR_WEIGHTS,
            disp_mean=disp_mean_t,
            disp_std=disp_std_t,
        )
        val_stats = run_phase1_epoch(
            pilot_model,
            val_loader_pilot,
            optimizer=None,
            physics_cfg=PHYSICS,
            green_load_fft=g_load_fft_t,
            green_poro_fft=g_poro_fft_t,
            weights=PRIOR_WEIGHTS,
            disp_mean=disp_mean_t,
            disp_std=disp_std_t,
        )
        row = {'epoch': epoch + 1, 'pilot_label': label}
        row.update({f'train_{k}': v for k, v in train_stats.items()})
        row.update({f'val_{k}': v for k, v in val_stats.items()})
        history.append(row)
        print(row)
        if val_stats['forward'] < best_val:
            best_val = val_stats['forward']
            torch.save(
                {
                    'epoch': epoch + 1,
                    'model_state_dict': pilot_model.state_dict(),
                    'optimizer_state_dict': pilot_optim.state_dict(),
                    'best_val_forward': best_val,
                    'normalization': {'x_mean': x_mean, 'x_std': x_std, 'obs_mean': obs_mean, 'obs_std': obs_std},
                    'tile_shape': (tile_h, tile_w),
                    'weights': PRIOR_WEIGHTS,
                    'pilot_group_config': pilot_group_config or {},
                    'learning_rate': learning_rate,
                    'init_checkpoint': str(init_checkpoint) if init_checkpoint is not None else None,
                },
                best_ckpt_path,
            )

    history_df = pd.DataFrame(history)
    history_path = OUT_DIR / f'punjab_phase1_pilot_history_{label}.csv'
    history_df.to_csv(history_path, index=False)
    summary = history_df.iloc[-1].to_dict()
    summary['best_val_forward'] = best_val
    summary['checkpoint'] = str(best_ckpt_path)
    summary['pilot_group_config'] = pilot_group_config or {}
    summary['learning_rate'] = learning_rate
    summary['init_checkpoint'] = str(init_checkpoint) if init_checkpoint is not None else None
    summary_path = OUT_DIR / f'punjab_phase1_pilot_summary_{label}.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    return pilot_model, history_df, history_path, summary_path, best_ckpt_path


grouped_pilot_model, grouped_pilot_history_df, grouped_pilot_history_path, grouped_pilot_summary_path, grouped_pilot_ckpt_path = run_phase1_pilot_train(
    train_loader_pilot=pilot_train_loader,
    val_loader_pilot=pilot_val_loader,
    label='grouped_support',
    pilot_group_config=PILOT_GROUP_CONFIG,
)
print('Saved grouped pilot history to:', grouped_pilot_history_path)
print('Saved grouped pilot summary to:', grouped_pilot_summary_path)
print('Saved grouped pilot checkpoint to:', grouped_pilot_ckpt_path)

expanded_pilot_model, expanded_pilot_history_df, expanded_pilot_history_path, expanded_pilot_summary_path, expanded_pilot_ckpt_path = run_phase1_pilot_train(
    train_loader_pilot=expanded_pilot_train_loader,
    val_loader_pilot=expanded_pilot_val_loader,
    label='grouped_support_expanded',
    pilot_group_config=EXPANDED_PILOT_GROUP_CONFIG,
)
print('Saved expanded grouped pilot history to:', expanded_pilot_history_path)
print('Saved expanded grouped pilot summary to:', expanded_pilot_summary_path)
print('Saved expanded grouped pilot checkpoint to:', expanded_pilot_ckpt_path)

fuller_phase1_model, fuller_phase1_history_df, fuller_phase1_history_path, fuller_phase1_summary_path, fuller_phase1_ckpt_path = run_phase1_pilot_train(
    train_loader_pilot=fuller_train_loader,
    val_loader_pilot=fuller_val_loader,
    label='phase1_fuller',
    pilot_group_config=FULLER_PHASE1_CONFIG,
    num_epochs=FULLER_PHASE1_CONFIG['num_epochs'],
)
print('Saved fuller Phase 1 history to:', fuller_phase1_history_path)
print('Saved fuller Phase 1 summary to:', fuller_phase1_summary_path)
print('Saved fuller Phase 1 checkpoint to:', fuller_phase1_ckpt_path)

finetune_phase1_model, finetune_phase1_history_df, finetune_phase1_history_path, finetune_phase1_summary_path, finetune_phase1_ckpt_path = run_phase1_pilot_train(
    train_loader_pilot=fuller_train_loader,
    val_loader_pilot=fuller_val_loader,
    label='phase1_fuller_finetune',
    pilot_group_config=FINETUNE_PHASE1_CONFIG,
    num_epochs=FINETUNE_PHASE1_CONFIG['num_epochs'],
    init_checkpoint=expanded_pilot_ckpt_path,
    learning_rate=FINETUNE_PHASE1_CONFIG['learning_rate'],
)
print('Saved fuller fine-tune history to:', finetune_phase1_history_path)
print('Saved fuller fine-tune summary to:', finetune_phase1_summary_path)
print('Saved fuller fine-tune checkpoint to:', finetune_phase1_ckpt_path)

# Write stable fuller-run artifact names alongside the label-specific outputs.
fuller_phase1_history_df.to_csv(OUT_DIR / 'punjab_phase1_fuller_history.csv', index=False)
with open(fuller_phase1_summary_path) as f:
    fuller_phase1_summary = json.load(f)
with open(OUT_DIR / 'punjab_phase1_fuller_summary.json', 'w') as f:
    json.dump(fuller_phase1_summary, f, indent=2)
torch.save(torch.load(fuller_phase1_ckpt_path, map_location='cpu'), OUT_DIR / 'punjab_phase1_fuller_best.pt')

finetune_phase1_history_df.to_csv(OUT_DIR / 'punjab_phase1_fuller_finetune_history.csv', index=False)
with open(finetune_phase1_summary_path) as f:
    finetune_phase1_summary = json.load(f)
with open(OUT_DIR / 'punjab_phase1_fuller_finetune_summary.json', 'w') as f:
    json.dump(finetune_phase1_summary, f, indent=2)
torch.save(torch.load(finetune_phase1_ckpt_path, map_location='cpu'), OUT_DIR / 'punjab_phase1_fuller_finetune_best.pt')

# Refresh canonical pilot outputs to the best grouped-support variant available in this run.
with open(grouped_pilot_summary_path) as f:
    grouped_summary = json.load(f)
with open(expanded_pilot_summary_path) as f:
    expanded_summary = json.load(f)
if expanded_summary['best_val_forward'] <= grouped_summary['best_val_forward']:
    best_history_df = expanded_pilot_history_df
    best_summary = expanded_summary
    best_ckpt_path = expanded_pilot_ckpt_path
    best_label = 'grouped_support_expanded'
else:
    best_history_df = grouped_pilot_history_df
    best_summary = grouped_summary
    best_ckpt_path = grouped_pilot_ckpt_path
    best_label = 'grouped_support'

best_history_df.to_csv(OUT_DIR / 'punjab_phase1_pilot_history.csv', index=False)
with open(OUT_DIR / 'punjab_phase1_pilot_summary.json', 'w') as f:
    json.dump(best_summary, f, indent=2)
torch.save(torch.load(best_ckpt_path, map_location='cpu'), OUT_DIR / 'punjab_phase1_pilot_best.pt')
print('canonical pilot now points to:', best_label)

comparison_rows = []
for label, path in [
    ('revised_support_bounded', OUT_DIR / 'punjab_phase1_pilot_summary_revised_support.json'),
    ('grouped_support_temporal', grouped_pilot_summary_path),
    ('grouped_support_expanded', expanded_pilot_summary_path),
    ('phase1_fuller', fuller_phase1_summary_path),
    ('phase1_fuller_finetune', finetune_phase1_summary_path),
]:
    if Path(path).exists():
        with open(path) as f:
            summary = json.load(f)
        comparison_rows.append({
            'variant': label,
            'val_forward': summary['val_forward'],
            'val_total': summary['val_total'],
            'train_forward': summary['train_forward'],
            'best_val_forward': summary['best_val_forward'],
        })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(OUT_DIR / 'punjab_phase1_pilot_grouped_comparison.csv', index=False)
with open(OUT_DIR / 'punjab_phase1_pilot_grouped_comparison.json', 'w') as f:
    json.dump(comparison_rows, f, indent=2)
print(comparison_df.to_string(index=False))

pilot_model = expanded_pilot_model if best_label == 'grouped_support_expanded' else grouped_pilot_model
pilot_history_df = best_history_df
pilot_history_path = OUT_DIR / 'punjab_phase1_pilot_history.csv'
pilot_summary_path = OUT_DIR / 'punjab_phase1_pilot_summary.json'
pilot_ckpt_path = OUT_DIR / 'punjab_phase1_pilot_best.pt'









## Forward-Loss Diagnostics And Smoke Residuals

Inspect the first smoke-run prediction in deformation space. This is the quickest way to see whether the current forward term is calibrated well enough for a longer run.

In [ ]:
def collect_batch_diagnostics(model, loader, *, physics_cfg, green_load_fft, green_poro_fft, disp_mean, disp_std, max_batches=1):
    examples = []
    raw_rmse = []
    norm_rmse = []
    model.eval()
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            xb = batch['x'].to(DEVICE)
            obs_disp = batch['obs_disp'].to(DEVICE)
            mask = batch['mask'].to(DEVICE)
            pred_state = model(xb) * mask
            pred_disp = forward_two_layer_torch(pred_state, green_load_fft, green_poro_fft, physics_cfg) * mask
            residual = pred_disp - obs_disp
            raw_rmse.append(torch.sqrt(F.mse_loss(pred_disp, obs_disp)).item())
            norm_rmse.append(torch.sqrt(F.mse_loss(normalize_field(pred_disp, disp_mean, disp_std), normalize_field(obs_disp, disp_mean, disp_std))).item())
            examples.append({
                'obs_disp': batch['obs_disp'][0, 0].cpu().numpy(),
                'pred_disp': pred_disp[0, 0].cpu().numpy(),
                'residual': residual[0, 0].cpu().numpy(),
                'pred_s0': pred_state[0, 0].cpu().numpy(),
                'pred_sg': pred_state[0, 1].cpu().numpy(),
                'tile_row': int(batch['tile_row'][0].item()),
                'tile_col': int(batch['tile_col'][0].item()),
                'end_idx': int(batch['end_idx'][0].item()),
            })
            if (batch_idx + 1) >= max_batches:
                break
    return examples, {
        'forward_rmse_raw_mean': float(np.mean(raw_rmse)),
        'forward_rmse_raw_std': float(np.std(raw_rmse)),
        'forward_rmse_norm_mean': float(np.mean(norm_rmse)),
        'forward_rmse_norm_std': float(np.std(norm_rmse)),
        'n_batches': len(raw_rmse),
    }


def save_example_gallery(examples, out_path, title_prefix, max_examples=3):
    n = min(len(examples), max_examples)
    fig, axes = plt.subplots(n, 5, figsize=(20, 5 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)
    for i in range(n):
        ex = examples[i]
        panels = [
            ('Obs', ex['obs_disp'], 'coolwarm'),
            ('Pred', ex['pred_disp'], 'coolwarm'),
            ('Residual', ex['residual'], 'coolwarm'),
            ('Pred S0', ex['pred_s0'], 'viridis'),
            ('Pred Sg', ex['pred_sg'], 'viridis'),
        ]
        for j, (name, arr, cmap) in enumerate(panels):
            axes[i, j].imshow(arr, cmap=cmap)
            axes[i, j].set_title(f"{title_prefix} {i+1}: {name}")
            axes[i, j].set_xticks([])
            axes[i, j].set_yticks([])
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


def summarize_example_metadata(examples, diag_summary):
    return {
        'n_examples': len(examples),
        'tiles': [
            {'tile_row': ex['tile_row'], 'tile_col': ex['tile_col'], 'end_idx': ex['end_idx']}
            for ex in examples
        ],
        **diag_summary,
    }


smoke_examples, smoke_diag_summary = collect_batch_diagnostics(
    smoke_model,
    val_loader,
    physics_cfg=PHYSICS,
    green_load_fft=g_load_fft_t,
    green_poro_fft=g_poro_fft_t,
    disp_mean=disp_mean_t,
    disp_std=disp_std_t,
    max_batches=1,
)
grouped_examples, grouped_diag_summary = collect_batch_diagnostics(
    grouped_pilot_model,
    pilot_val_loader,
    physics_cfg=PHYSICS,
    green_load_fft=g_load_fft_t,
    green_poro_fft=g_poro_fft_t,
    disp_mean=disp_mean_t,
    disp_std=disp_std_t,
    max_batches=6,
)
expanded_examples, expanded_diag_summary = collect_batch_diagnostics(
    expanded_pilot_model,
    expanded_pilot_val_loader,
    physics_cfg=PHYSICS,
    green_load_fft=g_load_fft_t,
    green_poro_fft=g_poro_fft_t,
    disp_mean=disp_mean_t,
    disp_std=disp_std_t,
    max_batches=6,
)
fuller_examples, fuller_diag_summary = collect_batch_diagnostics(
    fuller_phase1_model,
    fuller_val_loader,
    physics_cfg=PHYSICS,
    green_load_fft=g_load_fft_t,
    green_poro_fft=g_poro_fft_t,
    disp_mean=disp_mean_t,
    disp_std=disp_std_t,
    max_batches=6,
)
finetune_examples, finetune_diag_summary = collect_batch_diagnostics(
    finetune_phase1_model,
    fuller_val_loader,
    physics_cfg=PHYSICS,
    green_load_fft=g_load_fft_t,
    green_poro_fft=g_poro_fft_t,
    disp_mean=disp_mean_t,
    disp_std=disp_std_t,
    max_batches=6,
)

with open(OUT_DIR / 'punjab_phase1_smoke_diagnostics.json', 'w') as f:
    json.dump(smoke_diag_summary, f, indent=2)
with open(OUT_DIR / 'punjab_phase1_pilot_diagnostics_grouped_support.json', 'w') as f:
    json.dump(summarize_example_metadata(grouped_examples, grouped_diag_summary), f, indent=2)
with open(OUT_DIR / 'punjab_phase1_pilot_diagnostics_grouped_support_expanded.json', 'w') as f:
    json.dump(summarize_example_metadata(expanded_examples, expanded_diag_summary), f, indent=2)
with open(OUT_DIR / 'punjab_phase1_fuller_diagnostics.json', 'w') as f:
    json.dump(summarize_example_metadata(fuller_examples, fuller_diag_summary), f, indent=2)
with open(OUT_DIR / 'punjab_phase1_fuller_finetune_diagnostics.json', 'w') as f:
    json.dump(summarize_example_metadata(finetune_examples, finetune_diag_summary), f, indent=2)
with open(OUT_DIR / 'punjab_phase1_pilot_diagnostics.json', 'w') as f:
    json.dump(summarize_example_metadata(expanded_examples, expanded_diag_summary), f, indent=2)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
smoke_example = smoke_examples[0]
axes[0].imshow(smoke_example['obs_disp'], cmap='coolwarm')
axes[0].set_title('Smoke Obs')
axes[1].imshow(smoke_example['pred_disp'], cmap='coolwarm')
axes[1].set_title('Smoke Pred')
axes[2].imshow(smoke_example['residual'], cmap='coolwarm')
axes[2].set_title('Smoke Residual')
axes[3].imshow(smoke_example['pred_sg'], cmap='viridis')
axes[3].set_title('Smoke Pred Sg')
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.savefig(FIG_DIR / 'punjab_phase1_smoke_residuals.png', dpi=150, bbox_inches='tight')
plt.close(fig)

save_example_gallery(grouped_examples, FIG_DIR / 'punjab_phase1_pilot_residuals_grouped_support.png', 'Grouped')
save_example_gallery(expanded_examples, FIG_DIR / 'punjab_phase1_pilot_residuals_grouped_support_expanded.png', 'Expanded')
save_example_gallery(fuller_examples, FIG_DIR / 'punjab_phase1_fuller_residuals.png', 'Fuller')
save_example_gallery(finetune_examples, FIG_DIR / 'punjab_phase1_fuller_finetune_residuals.png', 'FineTune')
# Canonical pilot figure tracks the current best bounded pilot, which is the expanded grouped run.
save_example_gallery(expanded_examples, FIG_DIR / 'punjab_phase1_pilot_residuals.png', 'Expanded')

print('smoke diagnostics')
print(json.dumps(smoke_diag_summary, indent=2))
print('grouped pilot diagnostics')
print(json.dumps(grouped_diag_summary, indent=2))
print('expanded grouped pilot diagnostics')
print(json.dumps(expanded_diag_summary, indent=2))
print('fuller phase1 diagnostics')
print(json.dumps(fuller_diag_summary, indent=2))
print('fuller fine-tune diagnostics')
print(json.dumps(finetune_diag_summary, indent=2))





## Evaluation And Export

Summarize the first Punjab generic-prior inversion outputs here, then extend later with GRACE/W3RA/GPS comparisons in a separate integration phase.

In [ ]:
RESULTS_TEMPLATE = {
    'model_kind': MODEL_KIND,
    'phase': PRIOR_CONFIG['phase'],
    'prior_weights': PRIOR_WEIGHTS,
    'prior_config': PRIOR_CONFIG,
    'pilot_group_config': PILOT_GROUP_CONFIG,
    'expanded_pilot_group_config': EXPANDED_PILOT_GROUP_CONFIG,
    'fuller_phase1_config': FULLER_PHASE1_CONFIG,
    'finetune_phase1_config': FINETUNE_PHASE1_CONFIG,
    'artifacts': {
        'coverage_summary_csv': str(OUT_DIR / 'punjab_phase1_coverage_summary.csv'),
        'coverage_figure': str(FIG_DIR / 'punjab_phase1_coverage_diagnostics.png'),
        'support_strategy_sweep_csv': str(OUT_DIR / 'punjab_phase1_support_strategy_sweep.csv'),
        'split_summary_csv': str(OUT_DIR / 'punjab_phase1_split_summary.csv'),
        'tile_inventory_csv': str(OUT_DIR / 'punjab_phase1_tile_inventory.csv'),
        'normalization_json': str(OUT_DIR / 'punjab_phase1_normalization.json'),
        'smoke_history_csv': str(OUT_DIR / 'punjab_phase1_smoke_history.csv'),
        'smoke_summary_json': str(OUT_DIR / 'punjab_phase1_smoke_summary.json'),
        'smoke_diagnostics_json': str(OUT_DIR / 'punjab_phase1_smoke_diagnostics.json'),
        'smoke_residual_figure': str(FIG_DIR / 'punjab_phase1_smoke_residuals.png'),
        'pilot_history_csv': str(OUT_DIR / 'punjab_phase1_pilot_history.csv'),
        'pilot_summary_json': str(OUT_DIR / 'punjab_phase1_pilot_summary.json'),
        'pilot_checkpoint': str(OUT_DIR / 'punjab_phase1_pilot_best.pt'),
        'pilot_diagnostics_json': str(OUT_DIR / 'punjab_phase1_pilot_diagnostics.json'),
        'pilot_residual_figure': str(FIG_DIR / 'punjab_phase1_pilot_residuals.png'),
        'pilot_grouped_history_csv': str(OUT_DIR / 'punjab_phase1_pilot_history_grouped_support.csv'),
        'pilot_grouped_summary_json': str(OUT_DIR / 'punjab_phase1_pilot_summary_grouped_support.json'),
        'pilot_grouped_checkpoint': str(OUT_DIR / 'punjab_phase1_pilot_best_grouped_support.pt'),
        'pilot_grouped_diagnostics_json': str(OUT_DIR / 'punjab_phase1_pilot_diagnostics_grouped_support.json'),
        'pilot_grouped_residual_figure': str(FIG_DIR / 'punjab_phase1_pilot_residuals_grouped_support.png'),
        'pilot_grouped_expanded_history_csv': str(OUT_DIR / 'punjab_phase1_pilot_history_grouped_support_expanded.csv'),
        'pilot_grouped_expanded_summary_json': str(OUT_DIR / 'punjab_phase1_pilot_summary_grouped_support_expanded.json'),
        'pilot_grouped_expanded_checkpoint': str(OUT_DIR / 'punjab_phase1_pilot_best_grouped_support_expanded.pt'),
        'pilot_grouped_expanded_diagnostics_json': str(OUT_DIR / 'punjab_phase1_pilot_diagnostics_grouped_support_expanded.json'),
        'pilot_grouped_expanded_residual_figure': str(FIG_DIR / 'punjab_phase1_pilot_residuals_grouped_support_expanded.png'),
        'pilot_grouped_comparison_csv': str(OUT_DIR / 'punjab_phase1_pilot_grouped_comparison.csv'),
        'pilot_grouped_comparison_json': str(OUT_DIR / 'punjab_phase1_pilot_grouped_comparison.json'),
        'fuller_history_csv': str(OUT_DIR / 'punjab_phase1_fuller_history.csv'),
        'fuller_summary_json': str(OUT_DIR / 'punjab_phase1_fuller_summary.json'),
        'fuller_checkpoint': str(OUT_DIR / 'punjab_phase1_fuller_best.pt'),
        'fuller_diagnostics_json': str(OUT_DIR / 'punjab_phase1_fuller_diagnostics.json'),
        'fuller_residual_figure': str(FIG_DIR / 'punjab_phase1_fuller_residuals.png'),
        'fuller_finetune_history_csv': str(OUT_DIR / 'punjab_phase1_fuller_finetune_history.csv'),
        'fuller_finetune_summary_json': str(OUT_DIR / 'punjab_phase1_fuller_finetune_summary.json'),
        'fuller_finetune_checkpoint': str(OUT_DIR / 'punjab_phase1_fuller_finetune_best.pt'),
        'fuller_finetune_diagnostics_json': str(OUT_DIR / 'punjab_phase1_fuller_finetune_diagnostics.json'),
        'fuller_finetune_residual_figure': str(FIG_DIR / 'punjab_phase1_fuller_finetune_residuals.png'),
    },
}

print(json.dumps(RESULTS_TEMPLATE, indent=2))





## Reliability-Weighted Expanded Baseline

This branch keeps the current best Punjab Phase 1 baseline fixed and changes only the forward term.

The reliability proxy is built from coherence and temporal support:

\[
	ilde c(i,j)=\mathrm{clip}\left(rac{\mathrm{coh}(i,j)-	au_c}{1-	au_c},0,1ight),
\qquad
	ilde f(i,j)=\mathrm{clip}\left(rac{f_{\mathrm{valid}}(i,j)-	au_f}{1-	au_f},0,1ight),
\]

\[
w(i,j)=M(i,j)\left[\gamma + (1-\gamma)\left(	ilde c(i,j)	ilde f(i,j)ight)^{1/2}ight],
\]

where $M(i,j)$ is the selected support mask, $	au_c=0.20$, $	au_f=0.05$, and $\gamma=0.25$.

The weighted forward term becomes:

\[
L^{(w)}_{fwd}=rac{\sum_{i,j} w(i,j)\left(\hat d^*(i,j)-d^*(i,j)ight)^2}{\sum_{i,j} w(i,j)}.
\]

All other terms remain unchanged:

\[
L_{phase1}^{(w)}=
\lambda_f L^{(w)}_{fwd}
+ \lambda_{s0}L^{S_0}_{spatial}
+ \lambda_{sg}L^{S_g}_{spatial}
+ \lambda_{t0}L^{S_0}_{temporal}
+ \lambda_{tg}L^{S_g}_{temporal}.
\]


In [ ]:
RELIABILITY_WEIGHT_CONFIG = {
    'strategy': 'coherence_temporal_weighted_forward',
    'coherence_floor': 0.25,
    'power': 0.5,
    'train_label': 'grouped_support_expanded_reliability',
    'num_epochs': 4,
}


def build_reliability_map(coh_full, time_valid_fraction, support_mask, support_cfg, reliability_cfg):
    coh_scaled = np.clip(
        (coh_full - support_cfg['coherence_threshold']) / max(1.0 - support_cfg['coherence_threshold'], 1e-6),
        0.0,
        1.0,
    )
    time_scaled = np.clip(
        (time_valid_fraction - support_cfg['time_valid_fraction_threshold']) / max(1.0 - support_cfg['time_valid_fraction_threshold'], 1e-6),
        0.0,
        1.0,
    )
    coh_scaled = np.nan_to_num(coh_scaled, nan=0.0, posinf=0.0, neginf=0.0)
    time_scaled = np.nan_to_num(time_scaled, nan=0.0, posinf=0.0, neginf=0.0)
    joint = np.power(coh_scaled * time_scaled, reliability_cfg['power'])
    reliability = support_mask.astype(np.float32) * (
        reliability_cfg['coherence_floor'] + (1.0 - reliability_cfg['coherence_floor']) * joint
    )
    reliability = np.nan_to_num(reliability, nan=0.0, posinf=0.0, neginf=0.0)
    return reliability.astype(np.float32)


class ReliabilityWeightedDataset(Dataset):
    def __init__(self, base_dataset, reliability_map):
        self.base_dataset = base_dataset
        self.reliability_map = reliability_map.astype(np.float32)

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        r0 = int(item['tile_row'])
        c0 = int(item['tile_col'])
        h = int(item['mask'].shape[-2])
        w = int(item['mask'].shape[-1])
        reliability = self.reliability_map[r0:r0 + h, c0:c0 + w]
        return {
            **item,
            'reliability': torch.tensor(reliability[None, ...], dtype=torch.float32),
        }


def weighted_mse_loss(pred, target, weight, eps=1e-6):
    w = weight.expand_as(pred).clamp_min(0.0)
    err = (pred - target) ** 2
    denom = w.sum().clamp_min(eps)
    return (w * err).sum() / denom


def build_phase1_loss_weighted(pred_state, pred_disp, obs_disp, weights, forward_weight_map, previous_state=None, disp_mean=None, disp_std=None):
    generic_terms = build_generic_prior_terms(pred_state, previous_state=previous_state)
    if disp_mean is not None and disp_std is not None and FORWARD_NORMALIZE:
        pred_disp_cmp = normalize_field(pred_disp, disp_mean, disp_std)
        obs_disp_cmp = normalize_field(obs_disp, disp_mean, disp_std)
    else:
        pred_disp_cmp = pred_disp
        obs_disp_cmp = obs_disp
    loss_forward = weighted_mse_loss(pred_disp_cmp, obs_disp_cmp, forward_weight_map)
    total = (
        weights['forward'] * loss_forward
        + weights['spatial_s0'] * generic_terms['spatial_s0']
        + weights['spatial_sg'] * generic_terms['spatial_sg']
        + weights['temporal_s0'] * generic_terms['temporal_s0']
        + weights['temporal_sg'] * generic_terms['temporal_sg']
    )
    return {
        'total': total,
        'forward': loss_forward,
        **generic_terms,
    }


def run_phase1_epoch_weighted(model, loader, optimizer=None, *, physics_cfg, green_load_fft, green_poro_fft, weights, device=DEVICE, disp_mean=None, disp_std=None):
    training = optimizer is not None
    model.train(mode=training)
    state_cache = {}
    stats = {
        'total': 0.0,
        'forward': 0.0,
        'spatial_s0': 0.0,
        'spatial_sg': 0.0,
        'temporal_s0': 0.0,
        'temporal_sg': 0.0,
        'n_samples': 0,
    }

    for batch in loader:
        xb = batch['x'].to(device)
        obs_disp = batch['obs_disp'].to(device)
        mask = batch['mask'].to(device)
        reliability = batch['reliability'].to(device) * mask
        tile_rows = batch['tile_row'].tolist()
        tile_cols = batch['tile_col'].tolist()

        with torch.set_grad_enabled(training):
            pred_state = model(xb)
            batch_losses = []
            cache_updates = []

            for i in range(pred_state.shape[0]):
                key = (int(tile_rows[i]), int(tile_cols[i]))
                prev_state = state_cache.get(key)
                pred_i = pred_state[i:i+1] * mask[i:i+1]
                pred_disp_i = forward_two_layer_torch(pred_i, green_load_fft, green_poro_fft, physics_cfg) * mask[i:i+1]
                losses = build_phase1_loss_weighted(
                    pred_i,
                    pred_disp_i,
                    obs_disp[i:i+1],
                    weights,
                    reliability[i:i+1],
                    previous_state=prev_state,
                    disp_mean=disp_mean,
                    disp_std=disp_std,
                )
                batch_losses.append(losses['total'])
                for name in ['total', 'forward', 'spatial_s0', 'spatial_sg', 'temporal_s0', 'temporal_sg']:
                    stats[name] += float(losses[name].detach().cpu())
                stats['n_samples'] += 1
                cache_updates.append((key, pred_i.detach()))

            loss = torch.stack(batch_losses).mean()
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        for key, cached_pred in cache_updates:
            state_cache[key] = cached_pred

    denom = max(stats['n_samples'], 1)
    return {k: (v / denom if k != 'n_samples' else v) for k, v in stats.items()}


def run_phase1_weighted_pilot(*, train_loader_pilot, val_loader_pilot, label, pilot_group_config, reliability_stats, num_epochs=4):
    pilot_model = DualDecoderFrequencySeparatedSwinUNet3D(
        base_dim=32,
        time_patch=2,
        spatial_patch=4,
        num_heads=4,
    ).to(DEVICE)
    pilot_optim = torch.optim.AdamW(pilot_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history = []
    best_val = float('inf')
    best_ckpt_path = OUT_DIR / f'punjab_phase1_pilot_best_{label}.pt'
    for epoch in range(num_epochs):
        train_stats = run_phase1_epoch_weighted(
            pilot_model,
            train_loader_pilot,
            optimizer=pilot_optim,
            physics_cfg=PHYSICS,
            green_load_fft=g_load_fft_t,
            green_poro_fft=g_poro_fft_t,
            weights=PRIOR_WEIGHTS,
            disp_mean=disp_mean_t,
            disp_std=disp_std_t,
        )
        val_stats = run_phase1_epoch_weighted(
            pilot_model,
            val_loader_pilot,
            optimizer=None,
            physics_cfg=PHYSICS,
            green_load_fft=g_load_fft_t,
            green_poro_fft=g_poro_fft_t,
            weights=PRIOR_WEIGHTS,
            disp_mean=disp_mean_t,
            disp_std=disp_std_t,
        )
        row = {'epoch': epoch + 1, 'pilot_label': label}
        row.update({f'train_{k}': v for k, v in train_stats.items()})
        row.update({f'val_{k}': v for k, v in val_stats.items()})
        history.append(row)
        print(row)
        if val_stats['forward'] < best_val:
            best_val = val_stats['forward']
            torch.save(
                {
                    'epoch': epoch + 1,
                    'model_state_dict': pilot_model.state_dict(),
                    'optimizer_state_dict': pilot_optim.state_dict(),
                    'best_val_forward': best_val,
                    'normalization': {'x_mean': x_mean, 'x_std': x_std, 'obs_mean': obs_mean, 'obs_std': obs_std},
                    'weights': PRIOR_WEIGHTS,
                    'pilot_group_config': pilot_group_config,
                    'reliability_weight_config': RELIABILITY_WEIGHT_CONFIG,
                    'reliability_stats': reliability_stats,
                },
                best_ckpt_path,
            )
    history_df = pd.DataFrame(history)
    history_path = OUT_DIR / f'punjab_phase1_pilot_history_{label}.csv'
    history_df.to_csv(history_path, index=False)
    summary = history_df.iloc[-1].to_dict()
    summary['best_val_forward'] = best_val
    summary['checkpoint'] = str(best_ckpt_path)
    summary['pilot_group_config'] = pilot_group_config
    summary['reliability_weight_config'] = RELIABILITY_WEIGHT_CONFIG
    summary['reliability_stats'] = reliability_stats
    summary_path = OUT_DIR / f'punjab_phase1_pilot_summary_{label}.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    return pilot_model, history_df, history_path, summary_path, best_ckpt_path


reliability_map = build_reliability_map(coh_full, time_valid_fraction, support_mask, SUPPORT_CONFIG, RELIABILITY_WEIGHT_CONFIG)
reliability_stats = {
    'support_fraction': float((reliability_map > 0).mean()),
    'weight_min': float(reliability_map[support_mask > 0].min()),
    'weight_mean': float(reliability_map[support_mask > 0].mean()),
    'weight_max': float(reliability_map[support_mask > 0].max()),
}

expanded_pilot_train_ds_weighted = ReliabilityWeightedDataset(expanded_pilot_train_ds, reliability_map)
expanded_pilot_val_ds_weighted = ReliabilityWeightedDataset(expanded_pilot_val_ds, reliability_map)
expanded_pilot_train_loader_weighted = DataLoader(expanded_pilot_train_ds_weighted, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
expanded_pilot_val_loader_weighted = DataLoader(expanded_pilot_val_ds_weighted, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

weighted_label = RELIABILITY_WEIGHT_CONFIG['train_label']
weighted_pilot_model, weighted_pilot_history_df, weighted_pilot_history_path, weighted_pilot_summary_path, weighted_pilot_ckpt_path = run_phase1_weighted_pilot(
    train_loader_pilot=expanded_pilot_train_loader_weighted,
    val_loader_pilot=expanded_pilot_val_loader_weighted,
    label=weighted_label,
    pilot_group_config=EXPANDED_PILOT_GROUP_CONFIG,
    reliability_stats=reliability_stats,
    num_epochs=RELIABILITY_WEIGHT_CONFIG['num_epochs'],
)

weighted_examples, weighted_diag_summary = collect_batch_diagnostics(
    weighted_pilot_model,
    expanded_pilot_val_loader_weighted,
    physics_cfg=PHYSICS,
    green_load_fft=g_load_fft_t,
    green_poro_fft=g_poro_fft_t,
    disp_mean=disp_mean_t,
    disp_std=disp_std_t,
    max_batches=6,
)
with open(OUT_DIR / f'punjab_phase1_pilot_diagnostics_{weighted_label}.json', 'w') as f:
    json.dump(summarize_example_metadata(weighted_examples, weighted_diag_summary), f, indent=2)
save_example_gallery(weighted_examples, FIG_DIR / f'punjab_phase1_pilot_residuals_{weighted_label}.png', 'Weighted')

baseline_summary = json.load(open(OUT_DIR / 'punjab_phase1_pilot_summary_grouped_support_expanded.json'))
baseline_diag = json.load(open(OUT_DIR / 'punjab_phase1_pilot_diagnostics_grouped_support_expanded.json'))
weighted_summary = json.load(open(weighted_pilot_summary_path))
comparison_df = pd.DataFrame([
    {
        'run': 'grouped_support_expanded',
        'val_forward': baseline_summary['val_forward'],
        'best_val_forward': baseline_summary.get('best_val_forward', baseline_summary['val_forward']),
        'forward_rmse_norm_mean': baseline_diag['forward_rmse_norm_mean'],
    },
    {
        'run': weighted_label,
        'val_forward': weighted_summary['val_forward'],
        'best_val_forward': weighted_summary.get('best_val_forward', weighted_summary['val_forward']),
        'forward_rmse_norm_mean': weighted_diag_summary['forward_rmse_norm_mean'],
    },
])
comparison_path = OUT_DIR / 'punjab_phase1_reliability_weight_comparison.csv'
comparison_df.to_csv(comparison_path, index=False)
with open(OUT_DIR / 'punjab_phase1_reliability_weight_comparison.json', 'w') as f:
    json.dump(comparison_df.to_dict(orient='records'), f, indent=2)
with open(OUT_DIR / 'punjab_phase1_reliability_weight_stats.json', 'w') as f:
    json.dump(reliability_stats, f, indent=2)

print('reliability stats')
print(json.dumps(reliability_stats, indent=2))
print('saved weighted history to:', weighted_pilot_history_path)
print('saved weighted summary to:', weighted_pilot_summary_path)
print('saved weighted diagnostics to:', OUT_DIR / f'punjab_phase1_pilot_diagnostics_{weighted_label}.json')
print('saved weighted comparison to:', comparison_path)
print(comparison_df.to_string(index=False))


## Reliability-Weighted Results Summary

The reliability-weighted forward branch kept the expanded grouped-support sampler fixed and changed only the forward term:

\[
L^{(w)}_{fwd}=rac{\sum_{i,j} w(i,j)\left(\hat d^*(i,j)-d^*(i,j)ight)^2}{\sum_{i,j} w(i,j)}.
\]

In practice this branch was numerically unstable in the current Phase 1 setup. After fixing an initial `NaN` propagation bug in the reliability map construction, the weighted run still collapsed to `NaN` training and validation losses.

Current interpretation:
- the expanded grouped-support baseline remains the best Phase 1 Punjab baseline
- simple reliability weighting is too aggressive for the present optimization setup
- the next change should target a milder optimization or loss rebalance rather than replacing the baseline

Primary artifacts:
- `outputs/punjab_prior/punjab_phase1_pilot_summary_grouped_support_expanded_reliability.json`
- `outputs/punjab_prior/punjab_phase1_pilot_diagnostics_grouped_support_expanded_reliability.json`
- `outputs/punjab_prior/punjab_phase1_reliability_weight_comparison.csv`
- `outputs/punjab_prior/punjab_phase1_reliability_weight_stats.json`


## Rebalanced Expanded Baseline

This branch keeps the current best expanded grouped-support sampler fixed and makes a milder optimization change than direct reliability weighting.

The inverse map and forward model remain:

\[
(\hat S_0,\hat S_g)=f_	heta(d),
\qquad
\hat d = G_{\mathrm{load}}(\hat S_0) + G_{\mathrm{poro}}(\hat S_g).
\]

The modified objective is:

\[
L^{(reb)}_{phase1}=
\lambda_f L_{fwd}
+ \lambda_{s0}L^{S_0}_{spatial}
+ \lambda_{sg}L^{S_g}_{spatial}
+ \lambda_{t0}L^{S_0}_{temporal}
+ \lambda_{tg}L^{S_g}_{temporal},
\]

with the same normalized forward term,

\[
L_{fwd}=\mathrm{MSE}(\hat d^*, d^*),
\]

but using a warmer start and slightly stronger generic priors so they contribute meaningfully to training:

\[
	heta_0 \leftarrow 	heta_{\mathrm{expanded}},
\qquad
\eta = 5	imes 10^{-5}.
\]


In [ ]:
REBALANCED_PHASE1_CONFIG = {
    'strategy': 'grouped_support_expanded_rebalanced',
    'init_from': 'expanded_grouped_support_best',
    'learning_rate': 5e-5,
    'num_epochs': 4,
    'weights': {
        'forward': 1.0,
        'spatial_s0': 5.0,
        'spatial_sg': 5.0,
        'temporal_s0': 1.0,
        'temporal_sg': 1.0,
        'hydrology': 0.0,
        'grace': 0.0,
        'gps': 0.0,
    },
}


def run_phase1_pilot_train_custom_weights(*, train_loader_pilot, val_loader_pilot, label, pilot_group_config, weights, num_epochs=4, init_checkpoint=None, learning_rate=LEARNING_RATE):
    pilot_model = DualDecoderFrequencySeparatedSwinUNet3D(
        base_dim=32,
        time_patch=2,
        spatial_patch=4,
        num_heads=4,
    ).to(DEVICE)
    pilot_optim = torch.optim.AdamW(pilot_model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    if init_checkpoint is not None and Path(init_checkpoint).exists():
        ckpt = torch.load(init_checkpoint, map_location=DEVICE)
        pilot_model.load_state_dict(ckpt['model_state_dict'])
    history = []
    best_val = float('inf')
    best_ckpt_path = OUT_DIR / f'punjab_phase1_pilot_best_{label}.pt'
    for epoch in range(num_epochs):
        train_stats = run_phase1_epoch(
            pilot_model,
            train_loader_pilot,
            optimizer=pilot_optim,
            physics_cfg=PHYSICS,
            green_load_fft=g_load_fft_t,
            green_poro_fft=g_poro_fft_t,
            weights=weights,
            disp_mean=disp_mean_t,
            disp_std=disp_std_t,
        )
        val_stats = run_phase1_epoch(
            pilot_model,
            val_loader_pilot,
            optimizer=None,
            physics_cfg=PHYSICS,
            green_load_fft=g_load_fft_t,
            green_poro_fft=g_poro_fft_t,
            weights=weights,
            disp_mean=disp_mean_t,
            disp_std=disp_std_t,
        )
        row = {'epoch': epoch + 1, 'pilot_label': label}
        row.update({f'train_{k}': v for k, v in train_stats.items()})
        row.update({f'val_{k}': v for k, v in val_stats.items()})
        history.append(row)
        print(row)
        if val_stats['forward'] < best_val:
            best_val = val_stats['forward']
            torch.save(
                {
                    'epoch': epoch + 1,
                    'model_state_dict': pilot_model.state_dict(),
                    'optimizer_state_dict': pilot_optim.state_dict(),
                    'best_val_forward': best_val,
                    'normalization': {'x_mean': x_mean, 'x_std': x_std, 'obs_mean': obs_mean, 'obs_std': obs_std},
                    'weights': weights,
                    'pilot_group_config': pilot_group_config,
                    'learning_rate': learning_rate,
                    'init_checkpoint': str(init_checkpoint) if init_checkpoint is not None else None,
                },
                best_ckpt_path,
            )
    history_df = pd.DataFrame(history)
    history_path = OUT_DIR / f'punjab_phase1_pilot_history_{label}.csv'
    history_df.to_csv(history_path, index=False)
    summary = history_df.iloc[-1].to_dict()
    summary['best_val_forward'] = best_val
    summary['checkpoint'] = str(best_ckpt_path)
    summary['pilot_group_config'] = pilot_group_config
    summary['weights'] = weights
    summary['learning_rate'] = learning_rate
    summary['init_checkpoint'] = str(init_checkpoint) if init_checkpoint is not None else None
    summary_path = OUT_DIR / f'punjab_phase1_pilot_summary_{label}.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    return pilot_model, history_df, history_path, summary_path, best_ckpt_path


rebalanced_label = REBALANCED_PHASE1_CONFIG['strategy']
rebalanced_model, rebalanced_history_df, rebalanced_history_path, rebalanced_summary_path, rebalanced_ckpt_path = run_phase1_pilot_train_custom_weights(
    train_loader_pilot=expanded_pilot_train_loader,
    val_loader_pilot=expanded_pilot_val_loader,
    label=rebalanced_label,
    pilot_group_config=EXPANDED_PILOT_GROUP_CONFIG,
    weights=REBALANCED_PHASE1_CONFIG['weights'],
    num_epochs=REBALANCED_PHASE1_CONFIG['num_epochs'],
    init_checkpoint=expanded_pilot_ckpt_path,
    learning_rate=REBALANCED_PHASE1_CONFIG['learning_rate'],
)

rebalanced_examples, rebalanced_diag_summary = collect_batch_diagnostics(
    rebalanced_model,
    expanded_pilot_val_loader,
    physics_cfg=PHYSICS,
    green_load_fft=g_load_fft_t,
    green_poro_fft=g_poro_fft_t,
    disp_mean=disp_mean_t,
    disp_std=disp_std_t,
    max_batches=6,
)
with open(OUT_DIR / f'punjab_phase1_pilot_diagnostics_{rebalanced_label}.json', 'w') as f:
    json.dump(summarize_example_metadata(rebalanced_examples, rebalanced_diag_summary), f, indent=2)
save_example_gallery(rebalanced_examples, FIG_DIR / f'punjab_phase1_pilot_residuals_{rebalanced_label}.png', 'Rebalanced')

baseline_summary = json.load(open(OUT_DIR / 'punjab_phase1_pilot_summary_grouped_support_expanded.json'))
baseline_diag = json.load(open(OUT_DIR / 'punjab_phase1_pilot_diagnostics_grouped_support_expanded.json'))
rebalanced_summary = json.load(open(rebalanced_summary_path))
rebalanced_comparison_df = pd.DataFrame([
    {
        'run': 'grouped_support_expanded',
        'val_forward': baseline_summary['val_forward'],
        'best_val_forward': baseline_summary.get('best_val_forward', baseline_summary['val_forward']),
        'forward_rmse_norm_mean': baseline_diag['forward_rmse_norm_mean'],
    },
    {
        'run': rebalanced_label,
        'val_forward': rebalanced_summary['val_forward'],
        'best_val_forward': rebalanced_summary.get('best_val_forward', rebalanced_summary['val_forward']),
        'forward_rmse_norm_mean': rebalanced_diag_summary['forward_rmse_norm_mean'],
    },
])
rebalanced_comparison_path = OUT_DIR / 'punjab_phase1_rebalanced_comparison.csv'
rebalanced_comparison_df.to_csv(rebalanced_comparison_path, index=False)
with open(OUT_DIR / 'punjab_phase1_rebalanced_comparison.json', 'w') as f:
    json.dump(rebalanced_comparison_df.to_dict(orient='records'), f, indent=2)
print('saved rebalanced history to:', rebalanced_history_path)
print('saved rebalanced summary to:', rebalanced_summary_path)
print('saved rebalanced comparison to:', rebalanced_comparison_path)
print(rebalanced_comparison_df.to_string(index=False))


## Rebalanced Results Summary

The rebalanced expanded branch kept the current best sampler fixed, warm-started from the best expanded grouped-support checkpoint, and used a lower learning rate plus stronger generic-prior weights.

Observed result:
- forward fit stayed essentially unchanged relative to the expanded grouped-support baseline
- normalized forward RMSE remained the same
- spatial and temporal prior terms decreased modestly

Interpretation:
- the rebalanced branch is stable
- it does not improve deformation reconstruction over the current baseline
- it may produce slightly smoother states, but it is not a new best baseline on the primary Phase 1 metrics

Primary artifacts:
- `outputs/punjab_prior/punjab_phase1_pilot_summary_grouped_support_expanded_rebalanced.json`
- `outputs/punjab_prior/punjab_phase1_pilot_diagnostics_grouped_support_expanded_rebalanced.json`
- `outputs/punjab_prior/punjab_phase1_rebalanced_comparison.csv`
- `outputs/punjab_prior/figures/punjab_phase1_pilot_residuals_grouped_support_expanded_rebalanced.png`


## W3RA Prior Preparation

This section prepares the first external prior source without activating it in the Punjab loss yet.

For the current two-signal Punjab setup we align the clipped W3RA fields `S0` and `Sg` to Punjab acquisition dates by month:

\[
t_{\mathrm{month}} = \mathrm{month\_start}(t_{\mathrm{Punjab}}),
\qquad
(S_0^{W3RA}, S_g^{W3RA})(t_{\mathrm{Punjab}}) = (S_0, S_g)_{W3RA}(t_{\mathrm{month}}).
\]

We also build anomaly fields relative to the overlap-period mean:

\[
S_k^{\prime}(t) = S_k(t) - \overline{S_k}^{\,\mathrm{overlap}},
\qquad k \in \{0,g\}.
\]

At this stage these products are only aligned and exported. They are not yet active in the Phase 1 loss.


In [ ]:
from punjab_inversion.priors import (
    PriorAlignmentConfig,
    align_w3ra_to_punjab_dates,
    basin_mean_timeseries,
    compute_w3ra_anomalies,
    summarize_w3ra_alignment,
)

W3RA_PREP_CONFIG = {
    'variables': ('S0', 'Sg'),
    'baseline_mode': 'overlap_mean',
    'write_aligned_dataset': True,
}

alignment_cfg = PriorAlignmentConfig(baseline_mode=W3RA_PREP_CONFIG['baseline_mode'])

w3ra_aligned = align_w3ra_to_punjab_dates(
    w3ra_ref,
    punjab_meta['dates'],
    variables=W3RA_PREP_CONFIG['variables'],
)
w3ra_aligned_anom = compute_w3ra_anomalies(
    w3ra_aligned,
    baseline_start=punjab_meta['dates'].min(),
    baseline_end=punjab_meta['dates'].max(),
    variables=W3RA_PREP_CONFIG['variables'],
)

w3ra_summary = summarize_w3ra_alignment(
    w3ra_aligned,
    anomaly_ds=w3ra_aligned_anom,
    variables=W3RA_PREP_CONFIG['variables'],
)

w3ra_timeseries = basin_mean_timeseries(w3ra_aligned, variables=W3RA_PREP_CONFIG['variables'])
w3ra_timeseries_anom = basin_mean_timeseries(w3ra_aligned_anom, variables=W3RA_PREP_CONFIG['variables'])
for col in [c for c in w3ra_timeseries_anom.columns if c.endswith('_mean')]:
    w3ra_timeseries[f'{col}_anom'] = w3ra_timeseries_anom[col].values

aligned_path = OUT_DIR / 'punjab_phase2_w3ra_aligned.nc'
aligned_anom_path = OUT_DIR / 'punjab_phase2_w3ra_aligned_anomalies.nc'
summary_path = OUT_DIR / 'punjab_phase2_w3ra_alignment_summary.json'
table_path = OUT_DIR / 'punjab_phase2_w3ra_alignment_table.csv'
timeseries_path = OUT_DIR / 'punjab_phase2_w3ra_basin_timeseries.csv'
figure_path = FIG_DIR / 'punjab_phase2_w3ra_basin_timeseries.png'

if W3RA_PREP_CONFIG['write_aligned_dataset']:
    w3ra_aligned.to_netcdf(aligned_path)
    w3ra_aligned_anom.to_netcdf(aligned_anom_path)

with open(summary_path, 'w') as f:
    json.dump({**w3ra_summary, 'config': W3RA_PREP_CONFIG}, f, indent=2)
pd.DataFrame([w3ra_summary]).to_csv(table_path, index=False)
w3ra_timeseries.to_csv(timeseries_path, index=False)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(w3ra_timeseries['time'], w3ra_timeseries['S0_mean'], label='W3RA S0 mean', color='tab:blue')
axes[0].plot(w3ra_timeseries['time'], w3ra_timeseries['Sg_mean'], label='W3RA Sg mean', color='tab:green')
axes[0].set_title('Aligned W3RA basin-mean storage over Punjab dates')
axes[0].legend(loc='upper right')
axes[1].plot(w3ra_timeseries['time'], w3ra_timeseries['S0_mean_anom'], label='W3RA S0 anomaly', color='tab:blue')
axes[1].plot(w3ra_timeseries['time'], w3ra_timeseries['Sg_mean_anom'], label='W3RA Sg anomaly', color='tab:green')
axes[1].set_title('Aligned W3RA basin-mean anomalies over overlap mean')
axes[1].legend(loc='upper right')
plt.tight_layout()
plt.savefig(figure_path, dpi=150, bbox_inches='tight')
plt.close(fig)

print(json.dumps({**w3ra_summary, 'aligned_path': str(aligned_path), 'timeseries_path': str(timeseries_path)}, indent=2))


## Weak W3RA Prior Pilot

This branch activates the first external prior on top of the frozen Phase 1 baseline.

The sampler, support mask, and forward model remain fixed. The only new term is a weak hydrologic prior using aligned W3RA anomalies:

\[
L_{phase2,W3RA} = L_{phase1} + \lambda_{w} L_{W3RA}
\]

with

\[
L_{W3RA} = \mathrm{MSE}(\hat S, S^{W3RA}_{anom})
\]

where `S^{W3RA}_{anom}` contains aligned and interpolated W3RA anomaly tiles for `S0` and `Sg` at the Punjab acquisition dates.

This first test uses a warm start from the best expanded grouped-support checkpoint and keeps the W3RA term deliberately weak.


In [ ]:
from punjab_inversion.priors import interpolate_w3ra_tile

W3RA_PRIOR_CONFIG = {
    'strategy': 'grouped_support_expanded_w3ra_weak',
    'variables': ('S0', 'Sg'),
    'hydrology_weight': 0.05,
    'learning_rate': 5e-5,
    'num_epochs': 4,
    'interp_method': 'linear',
    'init_from': 'expanded_grouped_support_best',
}


class W3RAPriorDataset(Dataset):
    def __init__(self, base_dataset, w3ra_aligned_anom_ds, lat_values, lon_values, variables=('S0', 'Sg'), method='linear'):
        self.base_dataset = base_dataset
        self.w3ra_aligned_anom_ds = w3ra_aligned_anom_ds
        self.lat_values = np.asarray(lat_values)
        self.lon_values = np.asarray(lon_values)
        self.variables = tuple(variables)
        self.method = method
        self._cache = {}

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        key = (int(item['end_idx']), int(item['tile_row']), int(item['tile_col']))
        if key not in self._cache:
            end_idx, r0, c0 = key
            h = int(item['mask'].shape[-2])
            w = int(item['mask'].shape[-1])
            tile_target = interpolate_w3ra_tile(
                self.w3ra_aligned_anom_ds,
                punjab_meta['dates'][end_idx],
                self.lat_values[r0:r0 + h],
                self.lon_values[c0:c0 + w],
                variables=self.variables,
                method=self.method,
            )
            self._cache[key] = tile_target
        return {
            **item,
            'hydrology': torch.tensor(self._cache[key], dtype=torch.float32),
        }


def run_phase1_epoch_with_external(model, loader, optimizer=None, *, physics_cfg, green_load_fft, green_poro_fft, weights, device=DEVICE, disp_mean=None, disp_std=None):
    training = optimizer is not None
    model.train(mode=training)
    state_cache = {}
    stats = {
        'total': 0.0,
        'forward': 0.0,
        'spatial_s0': 0.0,
        'spatial_sg': 0.0,
        'temporal_s0': 0.0,
        'temporal_sg': 0.0,
        'hydrology': 0.0,
        'n_samples': 0,
    }

    for batch in loader:
        xb = batch['x'].to(device)
        obs_disp = batch['obs_disp'].to(device)
        mask = batch['mask'].to(device)
        hydrology = batch['hydrology'].to(device) * mask
        tile_rows = batch['tile_row'].tolist()
        tile_cols = batch['tile_col'].tolist()

        with torch.set_grad_enabled(training):
            pred_state = model(xb)
            batch_losses = []
            cache_updates = []

            for i in range(pred_state.shape[0]):
                key = (int(tile_rows[i]), int(tile_cols[i]))
                prev_state = state_cache.get(key)
                pred_i = pred_state[i:i+1] * mask[i:i+1]
                pred_disp_i = forward_two_layer_torch(pred_i, green_load_fft, green_poro_fft, physics_cfg) * mask[i:i+1]
                losses = build_phase1_loss(
                    pred_i,
                    pred_disp_i,
                    obs_disp[i:i+1],
                    weights,
                    previous_state=prev_state,
                    external_targets={'hydrology': hydrology[i:i+1]},
                    disp_mean=disp_mean,
                    disp_std=disp_std,
                )
                batch_losses.append(losses['total'])
                for name in ['total', 'forward', 'spatial_s0', 'spatial_sg', 'temporal_s0', 'temporal_sg', 'hydrology']:
                    stats[name] += float(losses[name].detach().cpu())
                stats['n_samples'] += 1
                cache_updates.append((key, pred_i.detach()))

            loss = torch.stack(batch_losses).mean()
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        for key, cached_pred in cache_updates:
            state_cache[key] = cached_pred

    denom = max(stats['n_samples'], 1)
    return {k: (v / denom if k != 'n_samples' else v) for k, v in stats.items()}


def run_w3ra_prior_pilot(*, train_loader_pilot, val_loader_pilot, label, weights, num_epochs, learning_rate, init_checkpoint):
    pilot_model = DualDecoderFrequencySeparatedSwinUNet3D(
        base_dim=32,
        time_patch=2,
        spatial_patch=4,
        num_heads=4,
    ).to(DEVICE)
    pilot_optim = torch.optim.AdamW(pilot_model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    if init_checkpoint is not None and Path(init_checkpoint).exists():
        ckpt = torch.load(init_checkpoint, map_location=DEVICE)
        pilot_model.load_state_dict(ckpt['model_state_dict'])
    history = []
    best_val = float('inf')
    best_ckpt_path = OUT_DIR / f'punjab_phase2_pilot_best_{label}.pt'
    for epoch in range(num_epochs):
        train_stats = run_phase1_epoch_with_external(
            pilot_model,
            train_loader_pilot,
            optimizer=pilot_optim,
            physics_cfg=PHYSICS,
            green_load_fft=g_load_fft_t,
            green_poro_fft=g_poro_fft_t,
            weights=weights,
            disp_mean=disp_mean_t,
            disp_std=disp_std_t,
        )
        val_stats = run_phase1_epoch_with_external(
            pilot_model,
            val_loader_pilot,
            optimizer=None,
            physics_cfg=PHYSICS,
            green_load_fft=g_load_fft_t,
            green_poro_fft=g_poro_fft_t,
            weights=weights,
            disp_mean=disp_mean_t,
            disp_std=disp_std_t,
        )
        row = {'epoch': epoch + 1, 'pilot_label': label}
        row.update({f'train_{k}': v for k, v in train_stats.items()})
        row.update({f'val_{k}': v for k, v in val_stats.items()})
        history.append(row)
        print(row)
        if val_stats['forward'] < best_val:
            best_val = val_stats['forward']
            torch.save(
                {
                    'epoch': epoch + 1,
                    'model_state_dict': pilot_model.state_dict(),
                    'optimizer_state_dict': pilot_optim.state_dict(),
                    'best_val_forward': best_val,
                    'weights': weights,
                    'learning_rate': learning_rate,
                    'init_checkpoint': str(init_checkpoint) if init_checkpoint is not None else None,
                    'w3ra_prior_config': W3RA_PRIOR_CONFIG,
                },
                best_ckpt_path,
            )
    history_df = pd.DataFrame(history)
    history_path = OUT_DIR / f'punjab_phase2_pilot_history_{label}.csv'
    history_df.to_csv(history_path, index=False)
    summary = history_df.iloc[-1].to_dict()
    summary['best_val_forward'] = best_val
    summary['checkpoint'] = str(best_ckpt_path)
    summary['weights'] = weights
    summary['learning_rate'] = learning_rate
    summary['init_checkpoint'] = str(init_checkpoint) if init_checkpoint is not None else None
    summary['w3ra_prior_config'] = W3RA_PRIOR_CONFIG
    summary_path = OUT_DIR / f'punjab_phase2_pilot_summary_{label}.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    return pilot_model, history_df, history_path, summary_path, best_ckpt_path


W3RA_PRIOR_WEIGHTS = {
    **PRIOR_WEIGHTS,
    'hydrology': W3RA_PRIOR_CONFIG['hydrology_weight'],
}

expanded_pilot_train_ds_w3ra = W3RAPriorDataset(
    expanded_pilot_train_ds,
    w3ra_aligned_anom,
    punjab_meta['lat'],
    punjab_meta['lon'],
    variables=W3RA_PRIOR_CONFIG['variables'],
    method=W3RA_PRIOR_CONFIG['interp_method'],
)
expanded_pilot_val_ds_w3ra = W3RAPriorDataset(
    expanded_pilot_val_ds,
    w3ra_aligned_anom,
    punjab_meta['lat'],
    punjab_meta['lon'],
    variables=W3RA_PRIOR_CONFIG['variables'],
    method=W3RA_PRIOR_CONFIG['interp_method'],
)
expanded_pilot_train_loader_w3ra = DataLoader(expanded_pilot_train_ds_w3ra, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
expanded_pilot_val_loader_w3ra = DataLoader(expanded_pilot_val_ds_w3ra, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

w3ra_label = W3RA_PRIOR_CONFIG['strategy']
w3ra_prior_model, w3ra_prior_history_df, w3ra_prior_history_path, w3ra_prior_summary_path, w3ra_prior_ckpt_path = run_w3ra_prior_pilot(
    train_loader_pilot=expanded_pilot_train_loader_w3ra,
    val_loader_pilot=expanded_pilot_val_loader_w3ra,
    label=w3ra_label,
    weights=W3RA_PRIOR_WEIGHTS,
    num_epochs=W3RA_PRIOR_CONFIG['num_epochs'],
    learning_rate=W3RA_PRIOR_CONFIG['learning_rate'],
    init_checkpoint=expanded_pilot_ckpt_path,
)

w3ra_examples, w3ra_diag_summary = collect_batch_diagnostics(
    w3ra_prior_model,
    expanded_pilot_val_loader_w3ra,
    physics_cfg=PHYSICS,
    green_load_fft=g_load_fft_t,
    green_poro_fft=g_poro_fft_t,
    disp_mean=disp_mean_t,
    disp_std=disp_std_t,
    max_batches=6,
)
with open(OUT_DIR / f'punjab_phase2_pilot_diagnostics_{w3ra_label}.json', 'w') as f:
    json.dump(summarize_example_metadata(w3ra_examples, w3ra_diag_summary), f, indent=2)
save_example_gallery(w3ra_examples, FIG_DIR / f'punjab_phase2_pilot_residuals_{w3ra_label}.png', 'W3RA')

baseline_summary = json.load(open(OUT_DIR / 'punjab_phase1_pilot_summary_grouped_support_expanded.json'))
baseline_diag = json.load(open(OUT_DIR / 'punjab_phase1_pilot_diagnostics_grouped_support_expanded.json'))
w3ra_summary_json = json.load(open(w3ra_prior_summary_path))
w3ra_comparison_df = pd.DataFrame([
    {
        'run': 'grouped_support_expanded',
        'val_forward': baseline_summary['val_forward'],
        'best_val_forward': baseline_summary.get('best_val_forward', baseline_summary['val_forward']),
        'forward_rmse_norm_mean': baseline_diag['forward_rmse_norm_mean'],
    },
    {
        'run': w3ra_label,
        'val_forward': w3ra_summary_json['val_forward'],
        'best_val_forward': w3ra_summary_json.get('best_val_forward', w3ra_summary_json['val_forward']),
        'forward_rmse_norm_mean': w3ra_diag_summary['forward_rmse_norm_mean'],
    },
])
w3ra_comparison_path = OUT_DIR / 'punjab_phase2_w3ra_comparison.csv'
w3ra_comparison_df.to_csv(w3ra_comparison_path, index=False)
with open(OUT_DIR / 'punjab_phase2_w3ra_comparison.json', 'w') as f:
    json.dump(w3ra_comparison_df.to_dict(orient='records'), f, indent=2)
print('saved W3RA prior history to:', w3ra_prior_history_path)
print('saved W3RA prior summary to:', w3ra_prior_summary_path)
print('saved W3RA prior comparison to:', w3ra_comparison_path)
print(w3ra_comparison_df.to_string(index=False))


## Weak W3RA Prior Results Summary

The first external-prior run kept the frozen expanded grouped-support baseline fixed and added a weak W3RA anomaly penalty:

\[
L_{phase2,W3RA} = L_{phase1} + \lambda_w L_{W3RA},
\qquad
\lambda_w = 0.05.
\]

Observed result:
- the run remained numerically stable
- forward-fit metrics were essentially unchanged relative to the frozen Phase 1 baseline
- the W3RA hydrology term decreased steadily during training, so the model did move slightly toward the W3RA anomaly target

Interpretation:
- a weak W3RA prior can be integrated without destabilizing the Punjab inversion
- at this weight it does not materially change the primary deformation-fit metrics
- this is a good Phase 2 starting point because it establishes a stable external-prior hook before stronger prior experiments

Primary artifacts:
- `outputs/punjab_prior/punjab_phase2_pilot_summary_grouped_support_expanded_w3ra_weak.json`
- `outputs/punjab_prior/punjab_phase2_pilot_diagnostics_grouped_support_expanded_w3ra_weak.json`
- `outputs/punjab_prior/punjab_phase2_w3ra_comparison.csv`
- `outputs/punjab_prior/figures/punjab_phase2_pilot_residuals_grouped_support_expanded_w3ra_weak.png`


## Stronger W3RA Prior Results Summary

A stronger W3RA anomaly prior was tested using the same setup as the weak W3RA branch, but with a larger hydrology weight:

\[
L_{phase2,W3RA} = L_{phase1} + \lambda_w L_{W3RA},
\qquad
\lambda_w = 0.20.
\]

Observed result:
- the stronger run stayed numerically stable
- forward-fit metrics again remained essentially unchanged relative to the frozen Phase 1 baseline
- the hydrology term decreased slightly further than in the weak W3RA run, so the model responded to the stronger prior without changing the main deformation fit

Interpretation:
- W3RA prior integration is stable across at least a modest weight range
- increasing the W3RA weight from `0.05` to `0.20` still does not materially change the primary Punjab Phase 2 fit metrics
- the next meaningful W3RA experiment would likely need a reformulation, not just another weight increase

Primary artifacts:
- `outputs/punjab_prior/punjab_phase2_pilot_summary_grouped_support_expanded_w3ra_stronger.json`
- `outputs/punjab_prior/punjab_phase2_w3ra_comparison.csv`


## GRACE Prior Preparation

This section prepares the GRACE/mascon external-prior path.

The target role of GRACE in the Punjab inversion is a coarse total-storage prior:

\[
L_{GRACE} = \mathrm{MSE}\left(A(\hat S_{tot}), TWS_{GRACE}ight),
\qquad
\hat S_{tot} = \hat S_0 + \hat S_g
\]

where $A(\cdot)$ is a coarse aggregation or alignment operator to the GRACE space.

At this stage we only:
1. search for local GRACE/mascon files,
2. record what is available,
3. and prepare the alignment hook.

If no local GRACE file is present, the section writes a clean discovery record and stops without pretending the prior is available.


In [ ]:
from punjab_inversion.priors import (
    GRACE_CANDIDATE_PATTERNS,
    PriorAlignmentConfig,
    align_grace_to_punjab_dates,
    basin_mean_timeseries,
    compute_grace_anomalies,
    discover_grace_candidates,
    summarize_grace_alignment,
)

GRACE_PREP_CONFIG = {
    'search_roots': [str(ROOT), '/mnt/data'],
    'variable': 'TWS',
    'baseline_mode': 'overlap_mean',
    'write_aligned_dataset': True,
}

candidate_paths = discover_grace_candidates(GRACE_PREP_CONFIG['search_roots'])
grace_discovery = {
    'search_roots': GRACE_PREP_CONFIG['search_roots'],
    'patterns': list(GRACE_CANDIDATE_PATTERNS),
    'n_candidates': len(candidate_paths),
    'candidates': candidate_paths[:50],
}

with open(OUT_DIR / 'punjab_phase2_grace_discovery.json', 'w') as f:
    json.dump(grace_discovery, f, indent=2)

if not candidate_paths:
    print(json.dumps(grace_discovery, indent=2))
    print('No local GRACE/mascon dataset found yet. Alignment hook is prepared but inactive.')
else:
    grace_path = candidate_paths[0]
    grace_ds = xr.open_dataset(grace_path)
    if GRACE_PREP_CONFIG['variable'] not in grace_ds.data_vars:
        raise ValueError(f"Configured GRACE variable {GRACE_PREP_CONFIG['variable']!r} not found in {grace_path}. Available: {list(grace_ds.data_vars)}")
    grace_aligned = align_grace_to_punjab_dates(
        grace_ds,
        punjab_meta['dates'],
        variable=GRACE_PREP_CONFIG['variable'],
    )
    grace_aligned_anom = compute_grace_anomalies(
        grace_aligned,
        variable=GRACE_PREP_CONFIG['variable'],
        baseline_start=punjab_meta['dates'].min(),
        baseline_end=punjab_meta['dates'].max(),
    )
    grace_summary = summarize_grace_alignment(
        grace_aligned,
        variable=GRACE_PREP_CONFIG['variable'],
        anomaly_ds=grace_aligned_anom,
    )
    grace_timeseries = basin_mean_timeseries(grace_aligned, variables=(GRACE_PREP_CONFIG['variable'],))
    grace_timeseries_anom = basin_mean_timeseries(grace_aligned_anom, variables=(GRACE_PREP_CONFIG['variable'],))
    grace_timeseries[f"{GRACE_PREP_CONFIG['variable']}_mean_anom"] = grace_timeseries_anom[f"{GRACE_PREP_CONFIG['variable']}_mean"].values

    aligned_path = OUT_DIR / 'punjab_phase2_grace_aligned.nc'
    aligned_anom_path = OUT_DIR / 'punjab_phase2_grace_aligned_anomalies.nc'
    summary_path = OUT_DIR / 'punjab_phase2_grace_alignment_summary.json'
    table_path = OUT_DIR / 'punjab_phase2_grace_alignment_table.csv'
    timeseries_path = OUT_DIR / 'punjab_phase2_grace_basin_timeseries.csv'
    figure_path = FIG_DIR / 'punjab_phase2_grace_basin_timeseries.png'

    if GRACE_PREP_CONFIG['write_aligned_dataset']:
        grace_aligned.to_netcdf(aligned_path)
        grace_aligned_anom.to_netcdf(aligned_anom_path)

    with open(summary_path, 'w') as f:
        json.dump({**grace_summary, 'config': GRACE_PREP_CONFIG, 'source_path': grace_path}, f, indent=2)
    pd.DataFrame([grace_summary]).to_csv(table_path, index=False)
    grace_timeseries.to_csv(timeseries_path, index=False)

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    axes[0].plot(grace_timeseries['time'], grace_timeseries[f"{GRACE_PREP_CONFIG['variable']}_mean"], color='tab:orange', label='GRACE mean')
    axes[0].set_title('Aligned GRACE basin-mean storage over Punjab dates')
    axes[0].legend(loc='upper right')
    axes[1].plot(grace_timeseries['time'], grace_timeseries[f"{GRACE_PREP_CONFIG['variable']}_mean_anom"], color='tab:red', label='GRACE anomaly')
    axes[1].set_title('Aligned GRACE basin-mean anomalies over overlap mean')
    axes[1].legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(figure_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

    print(json.dumps({**grace_summary, 'source_path': grace_path}, indent=2))


## GRACE Preparation Results Summary

The GRACE preparation block searched the configured local roots for a GRACE/mascon dataset and wrote a discovery record.

Observed result:
- no local GRACE/mascon dataset was found in the current workspace or `/mnt/data`
- the notebook therefore stopped at the discovery stage without trying to fake an alignment

Interpretation:
- the GRACE preparation hook is now in place
- the Punjab notebook is ready to align and export GRACE products as soon as a local GRACE/mascon file is added
- until that file exists, GRACE remains unavailable as an active Punjab prior

Primary artifact:
- `outputs/punjab_prior/punjab_phase2_grace_discovery.json`


## GRACE File Check And First Plot

This cell performs the first direct check of the local GRACE mascon file before any Punjab-prior alignment.

It does three things:
1. opens the local JPL mascon file,
2. subsets a Punjab-region bounding box,
3. plots a first diagnostic using:
   - basin-mean `lwe_thickness` and `uncertainty` time series over the Punjab overlap period,
   - overlap-mean regional maps for `lwe_thickness` and `uncertainty`.

This is only a data check. It does not yet activate a GRACE prior in the inversion loss.


In [ ]:
GRACE_CHECK_CONFIG = {
    'path': '/mnt/data/punjab_grace_mascon_l3/GRCTellus.JPL.200204_202512.GLO.RL06.3M.MSCNv04CRI.nc',
    'variable': 'lwe_thickness',
    'uncertainty_variable': 'uncertainty',
    'lat_min': 29.0,
    'lat_max': 33.0,
    'lon_min': 73.0,
    'lon_max': 77.0,
}

grace_path = Path(GRACE_CHECK_CONFIG['path'])
if not grace_path.exists():
    raise FileNotFoundError(f'GRACE file not found: {grace_path}')

grace_ds = xr.open_dataset(grace_path)
grace_region = grace_ds.sel(
    lat=slice(GRACE_CHECK_CONFIG['lat_min'], GRACE_CHECK_CONFIG['lat_max']),
    lon=slice(GRACE_CHECK_CONFIG['lon_min'], GRACE_CHECK_CONFIG['lon_max']),
)

grace_overlap = grace_region.sel(
    time=slice(pd.to_datetime(punjab_meta['dates'].min()), pd.to_datetime(punjab_meta['dates'].max()))
)

grace_timeseries = pd.DataFrame({
    'time': pd.to_datetime(grace_overlap.time.values),
    'lwe_thickness_mean': grace_overlap[GRACE_CHECK_CONFIG['variable']].mean(dim=('lat', 'lon')).values,
    'uncertainty_mean': grace_overlap[GRACE_CHECK_CONFIG['uncertainty_variable']].mean(dim=('lat', 'lon')).values,
})

grace_summary = {
    'path': str(grace_path),
    'n_times_total': int(grace_ds.sizes['time']),
    'n_times_overlap': int(grace_overlap.sizes['time']),
    'date_start_overlap': str(pd.to_datetime(grace_overlap.time.values[0]).date()),
    'date_end_overlap': str(pd.to_datetime(grace_overlap.time.values[-1]).date()),
    'lat_range': [float(grace_region.lat.min().item()), float(grace_region.lat.max().item())],
    'lon_range': [float(grace_region.lon.min().item()), float(grace_region.lon.max().item())],
    'lwe_mean_overlap': float(grace_overlap[GRACE_CHECK_CONFIG['variable']].mean().item()),
    'lwe_std_overlap': float(grace_overlap[GRACE_CHECK_CONFIG['variable']].std().item()),
    'uncertainty_mean_overlap': float(grace_overlap[GRACE_CHECK_CONFIG['uncertainty_variable']].mean().item()),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].plot(grace_timeseries['time'], grace_timeseries['lwe_thickness_mean'], color='tab:blue')
axes[0, 0].set_title('Punjab-region GRACE lwe_thickness mean')
axes[0, 1].plot(grace_timeseries['time'], grace_timeseries['uncertainty_mean'], color='tab:red')
axes[0, 1].set_title('Punjab-region GRACE uncertainty mean')
axes[1, 0].imshow(grace_overlap[GRACE_CHECK_CONFIG['variable']].mean(dim='time').values, cmap='coolwarm')
axes[1, 0].set_title('Overlap-mean lwe_thickness')
axes[1, 1].imshow(grace_overlap[GRACE_CHECK_CONFIG['uncertainty_variable']].mean(dim='time').values, cmap='magma')
axes[1, 1].set_title('Overlap-mean uncertainty')
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.savefig(FIG_DIR / 'punjab_phase2_grace_first_check.png', dpi=150, bbox_inches='tight')
plt.close(fig)

grace_timeseries.to_csv(OUT_DIR / 'punjab_phase2_grace_first_check_timeseries.csv', index=False)
with open(OUT_DIR / 'punjab_phase2_grace_first_check_summary.json', 'w') as f:
    json.dump(grace_summary, f, indent=2)

print(json.dumps(grace_summary, indent=2))
print('saved figure to:', FIG_DIR / 'punjab_phase2_grace_first_check.png')


## GRACE Alignment And Regional Series

This section aligns the JPL mascon product to Punjab acquisition dates by month and builds a coarse Punjab-region GRACE anomaly time series.

For the first Punjab GRACE prior we use the regional mean anomaly series:

\[
g_{GRACE}(t) = \mathrm{mean}_{\Omega_{Punjab}}\left(TWS_{GRACE}^{\prime}(t)ight)
\]

where

\[
TWS_{GRACE}^{\prime}(t)=TWS_{GRACE}(t)-\overline{TWS_{GRACE}}^{\,\mathrm{overlap}}.
\]

This produces a clean coarse target before any loss hook is activated.


In [ ]:
GRACE_ALIGN_CONFIG = {
    'path': '/mnt/data/punjab_grace_mascon_l3/GRCTellus.JPL.200204_202512.GLO.RL06.3M.MSCNv04CRI.nc',
    'variable': 'lwe_thickness',
    'uncertainty_variable': 'uncertainty',
    'lat_min': 29.0,
    'lat_max': 33.0,
    'lon_min': 73.0,
    'lon_max': 77.0,
    'write_aligned_dataset': True,
}

grace_ds = xr.open_dataset(GRACE_ALIGN_CONFIG['path'])
grace_region = grace_ds.sel(
    lat=slice(GRACE_ALIGN_CONFIG['lat_min'], GRACE_ALIGN_CONFIG['lat_max']),
    lon=slice(GRACE_ALIGN_CONFIG['lon_min'], GRACE_ALIGN_CONFIG['lon_max']),
)

grace_aligned = align_grace_to_punjab_dates(
    grace_region,
    punjab_meta['dates'],
    variable=GRACE_ALIGN_CONFIG['variable'],
)
grace_aligned_anom = compute_grace_anomalies(
    grace_aligned,
    variable=GRACE_ALIGN_CONFIG['variable'],
    baseline_start=punjab_meta['dates'].min(),
    baseline_end=punjab_meta['dates'].max(),
)

grace_alignment_summary = summarize_grace_alignment(
    grace_aligned,
    variable=GRACE_ALIGN_CONFIG['variable'],
    anomaly_ds=grace_aligned_anom,
)

grace_region_timeseries = basin_mean_timeseries(
    grace_aligned,
    variables=(GRACE_ALIGN_CONFIG['variable'],),
)
grace_region_timeseries_anom = basin_mean_timeseries(
    grace_aligned_anom,
    variables=(GRACE_ALIGN_CONFIG['variable'],),
)
grace_region_timeseries[f"{GRACE_ALIGN_CONFIG['variable']}_mean_anom"] = grace_region_timeseries_anom[f"{GRACE_ALIGN_CONFIG['variable']}_mean"].values

grace_aligned_path = OUT_DIR / 'punjab_phase2_grace_aligned.nc'
grace_aligned_anom_path = OUT_DIR / 'punjab_phase2_grace_aligned_anomalies.nc'
grace_alignment_summary_path = OUT_DIR / 'punjab_phase2_grace_alignment_summary.json'
grace_alignment_table_path = OUT_DIR / 'punjab_phase2_grace_alignment_table.csv'
grace_region_timeseries_path = OUT_DIR / 'punjab_phase2_grace_region_timeseries.csv'
grace_region_figure_path = FIG_DIR / 'punjab_phase2_grace_region_timeseries.png'

if GRACE_ALIGN_CONFIG['write_aligned_dataset']:
    grace_aligned.to_netcdf(grace_aligned_path)
    grace_aligned_anom.to_netcdf(grace_aligned_anom_path)

with open(grace_alignment_summary_path, 'w') as f:
    json.dump({**grace_alignment_summary, 'config': GRACE_ALIGN_CONFIG}, f, indent=2)
pd.DataFrame([grace_alignment_summary]).to_csv(grace_alignment_table_path, index=False)
grace_region_timeseries.to_csv(grace_region_timeseries_path, index=False)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(grace_region_timeseries['time'], grace_region_timeseries[f"{GRACE_ALIGN_CONFIG['variable']}_mean"], color='tab:blue')
axes[0].set_title('Aligned Punjab-region GRACE mean')
axes[1].plot(grace_region_timeseries['time'], grace_region_timeseries[f"{GRACE_ALIGN_CONFIG['variable']}_mean_anom"], color='tab:orange')
axes[1].set_title('Aligned Punjab-region GRACE anomaly mean')
plt.tight_layout()
plt.savefig(grace_region_figure_path, dpi=150, bbox_inches='tight')
plt.close(fig)

print(json.dumps({**grace_alignment_summary, 'aligned_path': str(grace_aligned_path)}, indent=2))


## Weak GRACE Prior Pilot

This first GRACE prior hook stays deliberately coarse and conservative.

We do **not** try to match each tile to the mascon field directly. Instead, for each batch we compare the predicted tile-mean total storage anomaly to the aligned Punjab-region GRACE anomaly series:

\[
\hat s_{tot,b} = \mathrm{mean}(\hat S_0 + \hat S_g)_b
\]

\[
L_{GRACE} = \mathrm{MSE}(z(\hat s_{tot,b}), z(g_{GRACE,b}))
\]

where `z(\cdot)` is batch standardization. The total loss is:

\[
L_{phase2,GRACE} = L_{phase1} + \lambda_g L_{GRACE}.
\]

This is only a first coarse pilot to test stability and influence.


In [ ]:
GRACE_PRIOR_CONFIG = {
    'strategy': 'grouped_support_expanded_grace_weak',
    'grace_weight': 0.05,
    'learning_rate': 5e-5,
    'num_epochs': 4,
    'init_from': 'expanded_grouped_support_best',
}


class GraceScalarDataset(Dataset):
    def __init__(self, base_dataset, grace_region_timeseries_df):
        self.base_dataset = base_dataset
        self.grace_lookup = {
            pd.Timestamp(row['time']): float(row[f"{GRACE_ALIGN_CONFIG['variable']}_mean_anom"])
            for _, row in grace_region_timeseries_df.iterrows()
        }

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        t = pd.Timestamp(punjab_meta['dates'][int(item['end_idx'])])
        return {
            **item,
            'grace_region_mean': torch.tensor(self.grace_lookup[t], dtype=torch.float32),
        }


def standardized_batch_mse(pred, target, eps=1e-6):
    if pred.numel() < 2:
        return torch.tensor(0.0, device=pred.device)
    pred_z = (pred - pred.mean()) / (pred.std(unbiased=False) + eps)
    target_z = (target - target.mean()) / (target.std(unbiased=False) + eps)
    return F.mse_loss(pred_z, target_z)


def run_phase1_epoch_with_grace(model, loader, optimizer=None, *, physics_cfg, green_load_fft, green_poro_fft, weights, device=DEVICE, disp_mean=None, disp_std=None):
    training = optimizer is not None
    model.train(mode=training)
    state_cache = {}
    stats = {
        'total': 0.0,
        'forward': 0.0,
        'spatial_s0': 0.0,
        'spatial_sg': 0.0,
        'temporal_s0': 0.0,
        'temporal_sg': 0.0,
        'grace': 0.0,
        'n_samples': 0,
    }

    for batch in loader:
        xb = batch['x'].to(device)
        obs_disp = batch['obs_disp'].to(device)
        mask = batch['mask'].to(device)
        grace_region_mean = batch['grace_region_mean'].to(device)
        tile_rows = batch['tile_row'].tolist()
        tile_cols = batch['tile_col'].tolist()

        with torch.set_grad_enabled(training):
            pred_state = model(xb)
            batch_losses = []
            cache_updates = []

            for i in range(pred_state.shape[0]):
                key = (int(tile_rows[i]), int(tile_cols[i]))
                prev_state = state_cache.get(key)
                pred_i = pred_state[i:i+1] * mask[i:i+1]
                pred_disp_i = forward_two_layer_torch(pred_i, green_load_fft, green_poro_fft, physics_cfg) * mask[i:i+1]
                losses = build_phase1_loss(
                    pred_i,
                    pred_disp_i,
                    obs_disp[i:i+1],
                    weights,
                    previous_state=prev_state,
                    disp_mean=disp_mean,
                    disp_std=disp_std,
                )
                batch_losses.append(losses['total'])
                for name in ['total', 'forward', 'spatial_s0', 'spatial_sg', 'temporal_s0', 'temporal_sg']:
                    stats[name] += float(losses[name].detach().cpu())
                stats['n_samples'] += 1
                cache_updates.append((key, pred_i.detach()))

            pred_total = (pred_state[:, 0:1] + pred_state[:, 1:2]) * mask
            mask_sum = mask.view(mask.shape[0], -1).sum(dim=1).clamp_min(1.0)
            pred_total_mean = pred_total.view(pred_total.shape[0], -1).sum(dim=1) / mask_sum
            grace_loss = standardized_batch_mse(pred_total_mean, grace_region_mean)
            total_loss = torch.stack(batch_losses).mean() + weights['grace'] * grace_loss

            if training:
                optimizer.zero_grad(set_to_none=True)
                total_loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        for key, cached_pred in cache_updates:
            state_cache[key] = cached_pred
        stats['grace'] += float(grace_loss.detach().cpu()) * pred_state.shape[0]
        stats['total'] += float((weights['grace'] * grace_loss).detach().cpu()) * pred_state.shape[0]

    denom = max(stats['n_samples'], 1)
    return {k: (v / denom if k != 'n_samples' else v) for k, v in stats.items()}


def run_grace_prior_pilot(*, train_loader_pilot, val_loader_pilot, label, weights, num_epochs, learning_rate, init_checkpoint):
    pilot_model = DualDecoderFrequencySeparatedSwinUNet3D(
        base_dim=32,
        time_patch=2,
        spatial_patch=4,
        num_heads=4,
    ).to(DEVICE)
    pilot_optim = torch.optim.AdamW(pilot_model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    if init_checkpoint is not None and Path(init_checkpoint).exists():
        ckpt = torch.load(init_checkpoint, map_location=DEVICE)
        pilot_model.load_state_dict(ckpt['model_state_dict'])
    history = []
    best_val = float('inf')
    best_ckpt_path = OUT_DIR / f'punjab_phase2_pilot_best_{label}.pt'
    for epoch in range(num_epochs):
        train_stats = run_phase1_epoch_with_grace(
            pilot_model,
            train_loader_pilot,
            optimizer=pilot_optim,
            physics_cfg=PHYSICS,
            green_load_fft=g_load_fft_t,
            green_poro_fft=g_poro_fft_t,
            weights=weights,
            disp_mean=disp_mean_t,
            disp_std=disp_std_t,
        )
        val_stats = run_phase1_epoch_with_grace(
            pilot_model,
            val_loader_pilot,
            optimizer=None,
            physics_cfg=PHYSICS,
            green_load_fft=g_load_fft_t,
            green_poro_fft=g_poro_fft_t,
            weights=weights,
            disp_mean=disp_mean_t,
            disp_std=disp_std_t,
        )
        row = {'epoch': epoch + 1, 'pilot_label': label}
        row.update({f'train_{k}': v for k, v in train_stats.items()})
        row.update({f'val_{k}': v for k, v in val_stats.items()})
        history.append(row)
        print(row)
        if val_stats['forward'] < best_val:
            best_val = val_stats['forward']
            torch.save(
                {
                    'epoch': epoch + 1,
                    'model_state_dict': pilot_model.state_dict(),
                    'optimizer_state_dict': pilot_optim.state_dict(),
                    'best_val_forward': best_val,
                    'weights': weights,
                    'learning_rate': learning_rate,
                    'init_checkpoint': str(init_checkpoint) if init_checkpoint is not None else None,
                    'grace_prior_config': GRACE_PRIOR_CONFIG,
                },
                best_ckpt_path,
            )
    history_df = pd.DataFrame(history)
    history_path = OUT_DIR / f'punjab_phase2_pilot_history_{label}.csv'
    history_df.to_csv(history_path, index=False)
    summary = history_df.iloc[-1].to_dict()
    summary['best_val_forward'] = best_val
    summary['checkpoint'] = str(best_ckpt_path)
    summary['weights'] = weights
    summary['learning_rate'] = learning_rate
    summary['init_checkpoint'] = str(init_checkpoint) if init_checkpoint is not None else None
    summary['grace_prior_config'] = GRACE_PRIOR_CONFIG
    summary_path = OUT_DIR / f'punjab_phase2_pilot_summary_{label}.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    return pilot_model, history_df, history_path, summary_path, best_ckpt_path


GRACE_PRIOR_WEIGHTS = {
    **PRIOR_WEIGHTS,
    'grace': GRACE_PRIOR_CONFIG['grace_weight'],
}

expanded_pilot_train_ds_grace = GraceScalarDataset(expanded_pilot_train_ds, grace_region_timeseries)
expanded_pilot_val_ds_grace = GraceScalarDataset(expanded_pilot_val_ds, grace_region_timeseries)
expanded_pilot_train_loader_grace = DataLoader(expanded_pilot_train_ds_grace, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
expanded_pilot_val_loader_grace = DataLoader(expanded_pilot_val_ds_grace, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

grace_label = GRACE_PRIOR_CONFIG['strategy']
grace_prior_model, grace_prior_history_df, grace_prior_history_path, grace_prior_summary_path, grace_prior_ckpt_path = run_grace_prior_pilot(
    train_loader_pilot=expanded_pilot_train_loader_grace,
    val_loader_pilot=expanded_pilot_val_loader_grace,
    label=grace_label,
    weights=GRACE_PRIOR_WEIGHTS,
    num_epochs=GRACE_PRIOR_CONFIG['num_epochs'],
    learning_rate=GRACE_PRIOR_CONFIG['learning_rate'],
    init_checkpoint=expanded_pilot_ckpt_path,
)

grace_examples, grace_diag_summary = collect_batch_diagnostics(
    grace_prior_model,
    expanded_pilot_val_loader_grace,
    physics_cfg=PHYSICS,
    green_load_fft=g_load_fft_t,
    green_poro_fft=g_poro_fft_t,
    disp_mean=disp_mean_t,
    disp_std=disp_std_t,
    max_batches=6,
)
with open(OUT_DIR / f'punjab_phase2_pilot_diagnostics_{grace_label}.json', 'w') as f:
    json.dump(summarize_example_metadata(grace_examples, grace_diag_summary), f, indent=2)
save_example_gallery(grace_examples, FIG_DIR / f'punjab_phase2_pilot_residuals_{grace_label}.png', 'GRACE')

baseline_summary = json.load(open(OUT_DIR / 'punjab_phase1_pilot_summary_grouped_support_expanded.json'))
baseline_diag = json.load(open(OUT_DIR / 'punjab_phase1_pilot_diagnostics_grouped_support_expanded.json'))
grace_summary_json = json.load(open(grace_prior_summary_path))
grace_comparison_df = pd.DataFrame([
    {
        'run': 'grouped_support_expanded',
        'val_forward': baseline_summary['val_forward'],
        'best_val_forward': baseline_summary.get('best_val_forward', baseline_summary['val_forward']),
        'forward_rmse_norm_mean': baseline_diag['forward_rmse_norm_mean'],
    },
    {
        'run': grace_label,
        'val_forward': grace_summary_json['val_forward'],
        'best_val_forward': grace_summary_json.get('best_val_forward', grace_summary_json['val_forward']),
        'forward_rmse_norm_mean': grace_diag_summary['forward_rmse_norm_mean'],
    },
])
grace_comparison_path = OUT_DIR / 'punjab_phase2_grace_comparison.csv'
grace_comparison_df.to_csv(grace_comparison_path, index=False)
with open(OUT_DIR / 'punjab_phase2_grace_comparison.json', 'w') as f:
    json.dump(grace_comparison_df.to_dict(orient='records'), f, indent=2)
print('saved GRACE prior history to:', grace_prior_history_path)
print('saved GRACE prior summary to:', grace_prior_summary_path)
print('saved GRACE prior comparison to:', grace_comparison_path)
print(grace_comparison_df.to_string(index=False))


## Stronger GRACE Prior Results Summary

The stronger GRACE pilot kept the same regional anomaly formulation and only increased the prior weight from $\lambda_g=0.05$ to $\lambda_g=0.20$.\n
$$L_{phase2,GRACE}=L_{phase1}+\lambda_g L_{\mathrm{GRACE}}$$

with

$$L_{\mathrm{GRACE}}=\mathrm{MSE}\left(z(\overline{\hat S_0+\hat S_g}),\; z(g_{\mathrm{GRACE}})\right).$$

Result: the run stayed stable but remained an effective tie with the frozen expanded grouped-support baseline on the main deformation-fit metrics.\n
Key outputs:
- `outputs/punjab_prior/punjab_phase2_pilot_summary_grouped_support_expanded_grace_stronger.json`
- `outputs/punjab_prior/punjab_phase2_grace_stronger_comparison.csv`
- `outputs/punjab_prior/figures/punjab_phase2_pilot_residuals_grouped_support_expanded_grace_stronger.png`


## Reformulated GRACE Prior Results Summary

This GRACE reformulation replaces the earlier batch-standardized level term with an uncertainty-weighted regional anomaly objective.\n
$$L_{phase2,GRACE}=L_{phase1}+\lambda_g L_{\mathrm{GRACE,uw}}$$

$$L_{\mathrm{GRACE,uw}}=\mathrm{mean}\left[\left(\frac{(\overline{\hat S_0+\hat S_g}-\mathrm{mean}(\overline{\hat S_0+\hat S_g}))-g_{\mathrm{GRACE}}}{\sigma_{\mathrm{GRACE}}}\right)^2\right].$$

Result: the run stayed stable and produced only a tiny numerical nudge relative to the frozen expanded grouped-support baseline, not a practically meaningful improvement.\n
Key outputs:
- `outputs/punjab_prior/punjab_phase2_pilot_summary_grouped_support_expanded_grace_uncertainty_anom.json`
- `outputs/punjab_prior/punjab_phase2_grace_reformulated_comparison.csv`
- `outputs/punjab_prior/figures/punjab_phase2_pilot_residuals_grouped_support_expanded_grace_uncertainty_anom.png`


## Interactive Baseline Inversion Map

This viewer uses the frozen Phase 1 expanded grouped-support baseline archive. The map is limited to the six reported baseline tiles across the full Punjab time span. Click a supported pixel to plot its predicted $S_0$ and $S_g$ time series.\n
If click events are not active in JupyterLab, run `%matplotlib widget` in a cell first.\n

In [ ]:
from punjab_inversion import launch_notebook_prediction_viewer

PHASE1_BASELINE_ARCHIVE = OUT_DIR / 'punjab_phase1_baseline_prediction_archive_expanded_tiles.h5'
if not PHASE1_BASELINE_ARCHIVE.exists():
    raise FileNotFoundError(f'Prediction archive not found: {PHASE1_BASELINE_ARCHIVE}')
print('If click events are not active, run `%matplotlib widget` in a notebook cell first.')
phase1_archive_viewer = launch_notebook_prediction_viewer(PHASE1_BASELINE_ARCHIVE)


## Baseline Export Plots

These cells read the saved Phase 1 baseline `nc` and `h5` exports directly. They produce paper-ready map snapshots and representative pixel time series from the frozen expanded grouped-support baseline.\n

In [ ]:
S0_NC_PATH = OUT_DIR / 'punjab_phase1_baseline_prediction_expanded_tiles_S0.nc'
SG_NC_PATH = OUT_DIR / 'punjab_phase1_baseline_prediction_expanded_tiles_Sg.nc'
ARCHIVE_PATH = OUT_DIR / 'punjab_phase1_baseline_prediction_archive_expanded_tiles.h5'

s0_nc = xr.open_dataset(S0_NC_PATH)
sg_nc = xr.open_dataset(SG_NC_PATH)
phase1_archive = PredictionArchive(ARCHIVE_PATH)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
panels = [
    ('S0 mean', s0_nc['S0_pred'].mean('time').values, 'viridis'),
    ('Sg mean', sg_nc['Sg_pred'].mean('time').values, 'viridis'),
    ('S0 last', s0_nc['S0_pred'].isel(time=-1).values, 'viridis'),
    ('Sg last', sg_nc['Sg_pred'].isel(time=-1).values, 'viridis'),
]
for ax, (title, arr, cmap) in zip(axes.ravel(), panels):
    im = ax.imshow(arr, cmap=cmap)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
baseline_export_map_path = FIG_DIR / 'punjab_phase1_baseline_export_maps.png'
plt.savefig(baseline_export_map_path, dpi=150, bbox_inches='tight')
plt.show()
print('saved maps to:', baseline_export_map_path)


In [ ]:
rows, cols = np.where(phase1_archive.support_mask)
sample_positions = [0, len(rows) // 2, len(rows) - 1]
sample_pixels = []
for idx in sample_positions:
    r = int(rows[idx]); c = int(cols[idx])
    series = phase1_archive.pixel_series(r, c)
    sample_pixels.append({'row': r, 'col': c, 'lat': float(s0_nc['lat'].values[r]), 'lon': float(s0_nc['lon'].values[c]), 'series': series})

fig, axes = plt.subplots(len(sample_pixels), 1, figsize=(12, 3.8 * len(sample_pixels)), sharex=True)
if len(sample_pixels) == 1:
    axes = [axes]
for ax, sample in zip(axes, sample_pixels):
    ax.plot(sample['series']['dates'], sample['series']['s0'], label='S0', color='tab:blue')
    ax.plot(sample['series']['dates'], sample['series']['sg'], label='Sg', color='tab:green')
    ax.set_title(f"row={sample['row']}, col={sample['col']} | lat={sample['lat']:.3f}, lon={sample['lon']:.3f}")
    ax.legend(loc='upper right')
    ax.grid(alpha=0.25)
axes[-1].set_xlabel('Date')
baseline_export_ts_path = FIG_DIR / 'punjab_phase1_baseline_export_timeseries.png'
plt.tight_layout()
plt.savefig(baseline_export_ts_path, dpi=150, bbox_inches='tight')
plt.show()
sample_pixel_table = pd.DataFrame([{k: v for k, v in sample.items() if k != 'series'} for sample in sample_pixels])
sample_pixel_table_path = OUT_DIR / 'punjab_phase1_baseline_export_sample_pixels.csv'
sample_pixel_table.to_csv(sample_pixel_table_path, index=False)
print('saved time series to:', baseline_export_ts_path)
print('saved sample pixels to:', sample_pixel_table_path)
phase1_archive.close()
